<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/ShortTrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade yfinance
!pip install finnhub-python
!pip install hurst -q
!pip install ta

In [3]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
import numpy as np
from datetime import datetime
import time
import ta
import random
import finnhub
import requests
from datetime import datetime, timedelta
from scipy.stats import linregress
from transformers import pipeline
import matplotlib.pyplot as plt
from hurst import compute_Hc

from dataclasses import dataclass, field
from typing import Optional
import warnings
warnings.filterwarnings("ignore")
# ensure reproducibility
random.seed(42)

# Display all rows
pd.set_option('display.max_rows', None)

# Display all columns
pd.set_option('display.max_columns', None)

# Prevent wrapping of wide DataFrames
pd.set_option('display.expand_frame_repr', False)

# Display full content of each cell
pd.set_option('display.max_colwidth', None)

# Display the full width of the DataFrame
pd.set_option('display.width', None)
print("Libraries Installed!")

1.7.0
Libraries Installed!


In [4]:
today = datetime.today()
start_of_year = today.replace(month=1, day=1)

# First day of this month
first_day_month = today.replace(day=1)

# First day of this week (Monday as weekday 0)
first_day_week = today - timedelta(days=today.weekday())
print("First day of year:", start_of_year)

print("\nFirst day of this month:", first_day_month)
print("\nFirst day of this week:", first_day_week)


def most_recent_quarter_start(today=None):
    if today is None:
        today = pd.Timestamp.today().normalize()
    year = today.year
    month = today.month

    # Determine quarter start months: Jan, Apr, Jul, Oct
    if month >= 10:
        q_start = pd.Timestamp(year, 10, 1)
    elif month >= 7:
        q_start = pd.Timestamp(year, 7, 1)
    elif month >= 4:
        q_start = pd.Timestamp(year, 4, 1)
    else:
        q_start = pd.Timestamp(year, 1, 1)

    return q_start

# Example usage
print("Today:", pd.Timestamp.today().normalize())
print("Most recent quarter start:", most_recent_quarter_start())

First day of year: 2026-01-01 02:53:53.493284

First day of this month: 2026-09-01 02:53:53.493284

First day of this week: 2026-08-31 02:53:53.493284
Today: 2026-09-03 00:00:00
Most recent quarter start: 2026-07-01 00:00:00


In [5]:
def get_last_quad_witching(reference_date=None):
    """
    Returns the most recent quad witching date (3rd Friday of Mar/Jun/Sep/Dec)
    before or equal to reference_date.
    """
    import calendar
    from datetime import date, timedelta

    if reference_date is None:
        reference_date = date.today()
    elif isinstance(reference_date, str):
        reference_date = pd.to_datetime(reference_date).date()

    quad_months = [3, 6, 9, 12]

    def third_friday(year, month):
        # Find first day of month
        first_day = date(year, month, 1)
        # Find first Friday
        first_friday = first_day + timedelta(days=(4 - first_day.weekday()) % 7)
        # Third Friday = first Friday + 14 days
        return first_friday + timedelta(days=14)

    # Generate last 2 years of quad witching dates
    candidates = []
    for year in [reference_date.year - 1, reference_date.year]:
        for month in quad_months:
            candidates.append(third_friday(year, month))

    # Filter to dates on or before reference_date
    past_dates = [d for d in candidates if d <= reference_date]

    # Return most recent
    return max(past_dates).strftime("%Y-%m-%d")

In [49]:
# List of ETFs to analyze
recent_quarter = most_recent_quarter_start()
#start_of_year = '2025-01-01'
df_raw = pd.read_csv('short_list.csv')
df_raw = df_raw[df_raw['Type'].isin(['ETF','Stock','ASX','TSX','AS'])]
#df_raw = df_raw[df_raw['Type'].isin(['TSX'])]
df_raw = df_raw.drop_duplicates(subset=['Asset'])
etfs = df_raw['Asset'].to_list()
print(etfs)

print(len(etfs))

['BURL', 'RBA', 'CW', 'BROS', 'BHF', 'MIDD', 'YETI', 'VVV', 'SARO', 'GME', 'FND', 'TKR', 'AAL', 'NYT', 'BYD', 'CHH', 'TNL', 'ALK', 'PII', 'CGNX', 'CRS', 'MUSA', 'ACM', 'POST', 'FOUR', 'GXO', 'MLI', 'FNF', 'CBT', 'R', 'KRG', 'SSD', 'PLNT', 'CROX', 'RYN', 'PINS', 'PFGC', 'MMS', 'HR', 'WWD', 'IBOC', 'FLG', 'NXST', 'BRX', 'BBWI', 'SFM', 'MOG.A', 'WPC', 'MTN', 'FCN', 'NJR', 'GHC', 'ADC', 'HXL', 'WMS', 'IDA', 'GBCI', 'MUR', 'CSL', 'JEF', 'GT', 'RPM', 'CYTK', 'TCBI', 'BCO', 'CUBE', 'EGP', 'LAMR', 'MSM', 'KBH', 'FFIN', 'FBIN', 'CTRE', 'NBIX', 'WTFC', 'KD', 'WAL', 'BRKR', 'CUZ', 'POR', 'VLY', 'TOL', 'SMG', 'UFPI', 'OHI', 'NNN', 'OZK', 'OGE', 'ELAN', 'FHN', 'KRC', 'PNFP', 'MAT', 'BDC', 'TEX', 'COLB', 'XRAY', 'ALLY', 'VNO', 'ELS', 'CR', 'AHR', 'AIT', 'AYI', 'BC', 'BJ', 'BLD', 'FN', 'FR', 'G', 'GTLS', 'H', 'JHG', 'M', 'NSA', 'OC', 'PB', 'PK', 'PR', 'RH', 'RS', 'SF', 'SN', 'SR', 'ST', 'STAG', 'TMHC', 'WBS', 'DCI', 'ENSG', 'IRT', 'AMH', 'EPR', 'PEN', 'FNB', 'SLAB', 'BSY', 'NWE', 'BKH', 'GGG', 'GATX'

In [50]:

def kaufman_er_pine(ticker, period=10, lookback="3mo"):
    data = yf.download(
        ticker,
        period=lookback,
        interval="1d",
        auto_adjust=False,   # match Pine
        progress=False
    )

    if data.empty:
        return np.nan

    close = data["Close"].squeeze()

    # Directional movement (matches Pine)
    direction = close.iloc[-1] - close.iloc[-period]

    # Total absolute movement (matches Pine)
    volatility = close.diff().abs().iloc[-period:].sum()

    if volatility == 0:
        return 0.0

    return (direction / volatility) * 100   # Pine scaling

def weinstein_stage(df, sma_window=30):
    """Determine Weinstein stage using 30-week SMA and its slope."""
     # Handle MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df["SMA"] = df["Close"].rolling(window=sma_window).mean()

    # Compute linear regression slope on last N SMA points
    if len(df.dropna()) < sma_window:
        return None  # not enough data

    slope, _, _, _, _ = linregress(range(5), df["SMA"].tail(5))

    latest_price = df["Close"].iloc[-1]
    latest_sma = df["SMA"].iloc[-1]

    # Determine stage
    if latest_price > latest_sma and slope > 0:
        stage = "Stage 2 (Advancing)"
    elif latest_price < latest_sma and slope < 0:
        stage = "Stage 4 (Declining)"
    elif  np.abs(slope) <= 0.001:
        stage = "Stage 1 (Basing)"
    else:
        stage = "Stage 3 (Topping)"

    return stage, slope, latest_price, latest_sma


In [51]:
# Classify stocks into stages
results = []
for etf in etfs:
    df = yf.download(etf, period="3y", interval="1wk", auto_adjust=True)
    stage_info = weinstein_stage(df)
    if stage_info:
        stage, slope, price, sma = stage_info
        results.append({
            "ETF": etf,
            "Stage": stage,
            "SMA_Slope": slope,
            "Latest_Price": price,
            "30W_SMA": sma
        })

stages_df = pd.DataFrame(results).sort_values(by="SMA_Slope", ascending=False)
declining_stocks= stages_df[stages_df["Stage"] .isin(["Stage 4 (Declining)"]) ]
declining_stocks.reset_index(drop=True, inplace=True)

# ensure negtaive stocks
df_o = df_raw[df_raw['Asset'].isin(declining_stocks['ETF'])]
df_raw = df_o.copy()
etfs2 = df_raw['Asset'].to_list()
etfss = list(dict.fromkeys(etfs2))

er_values = {
    ticker: kaufman_er_pine(ticker)
    for ticker in etfss
}

etfs= [
    ticker
    for ticker, er in er_values.items()
    if er < 0.0
]

print(len(etfs))

print(etfs)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

38
['RBA', 'CW', 'BROS', 'YETI', 'VVV', 'SARO', 'FND', 'AAL', 'BYD', 'PII', 'FOUR', 'MLI', 'FNF', 'PLNT', 'RYN', 'MMS', 'NXST', 'BBWI', 'FCN', 'GHC', 'WMS', 'GBCI', 'RPM', 'TCBI', 'BCO', 'KBH', 'FBIN', 'TOL', 'SMG', 'UFPI', 'BC', 'FN', 'RH', 'SR', 'ENSG', 'PEN', 'GGG', 'GATX']


# Brian Shannon daily timeframe stage classification

In [52]:

@dataclass
class ClassifierConfig:
    short_ma:       int   = 20
    medium_ma:      int   = 50
    long_ma:        int   = 200
    atr_period:     int   = 14
    slope_period:   int   = 10       # wider slope window = less noise
    rs_period:      int   = 63       # 3-month RS — more meaningful
    volume_period:  int   = 50       # 50-bar vol average — institutional grade
    pivot_lookback: int   = 10       # wider pivot = fewer false swings

    # Stage 2 thresholds
    stage2_min_score:       int   = 7
    stage2_ma50_slope_min:  float = 0.0
    stage2_ma200_slope_min: float = 0.0    # NEW — 200 MA must also be rising

    # Stage 1 thresholds — tighter = more accurate accumulation ID
    stage1_slope_max:       float = 0.03   # tighter than original 0.05
    stage1_atr_max:         float = 0.04   # tighter volatility compression
    stage1_ma50_proximity:  float = 0.07   # price within 7% of MA50

    # Stage 4 thresholds
    stage4_rs_penalty:      float = -0.05  # RS must be negative for hard Stage 4


# =============================================================
# CLASSIFIER
# =============================================================

class BrianShannonAuctionClassifier:
    """
    Improved Brian Shannon Auction Market Stage Classifier.

    Improvements over original:
    ----------------------------
    1.  Wider slope window (10 bars) — reduces slope noise
    2.  3-month RS period — more institutionally meaningful
    3.  MA200 slope condition added to Stage 2 — prevents false markups
    4.  Stage 1 uses tighter slope + ATR thresholds
    5.  Stage 4 requires negative RS — removes weak/sideways false declines
    6.  Weighted trend score — not all signals are equal
    7.  Stage confidence score added — tells you how strong each call is
    8.  Stage transition detection — flags when stage is changing
    9.  RS percentile rank — tells you where RS stands vs its own history
    10. Multi-stock scanner built in — scan a list and get ranked results
    """

    def __init__(self, config: Optional[ClassifierConfig] = None):
        self.config = config or ClassifierConfig()

    # =========================================================
    # PUBLIC — SINGLE STOCK
    # =========================================================
    def classify(self, df: pd.DataFrame) -> pd.DataFrame:

        df = df.copy()

        self._moving_averages(df)
        self._atr(df)
        self._relative_strength(df)
        self._volume_analysis(df)
        self._trend_structure(df)
        self._trend_score(df)
        self._classify_stages(df)
        self._stage_confidence(df)
        self._stage_transitions(df)

        return df

    # =========================================================
    # PUBLIC — MULTI STOCK SCANNER
    # =========================================================
    def scan(
        self,
        tickers:    list,
        benchmark:  str  = "SPY",
        start:      str  = "2021-01-01",
        min_score:  int  = 0,
        stage_filter: Optional[int] = None
    ) -> pd.DataFrame:
        """
        Scan a list of tickers and return a ranked summary DataFrame.

        Parameters
        ----------
        tickers      : list of ticker symbols
        benchmark    : benchmark ticker for RS (default SPY)
        start        : start date for data download
        min_score    : minimum trend_score to include in results
        stage_filter : filter by stage number (1/2/3/4) or None for all

        Returns
        -------
        pd.DataFrame sorted by trend_score descending
        """

        print(f"\nDownloading benchmark ({benchmark})...")
        spy_data = yf.download(benchmark, start=start, interval= '1d', progress=False)

        results = []

        for i, ticker in enumerate(tickers, 1):

            print(f"[{i}/{len(tickers)}] Processing {ticker}...", end=" ")

            try:

                data = yf.download(ticker, start=start, interval= '1d', progress=False)

                if data.empty or len(data) < self.config.long_ma + 10:
                    print("SKIP — insufficient data")
                    continue

                # flatten multi-level columns if present
                if isinstance(data.columns, pd.MultiIndex):
                    data.columns = data.columns.get_level_values(0)

                data["benchmark_close"] = spy_data["Close"].reindex(
                    data.index
                ).ffill()

                classified = self.classify(data)
                latest     = classified.iloc[-1]
                prev       = classified.iloc[-2]

                results.append({
                    "Ticker":        ticker,
                    "Close":         round(latest["Close"], 2),
                    "Stage":         int(latest["stage_number"])
                                     if not np.isnan(latest["stage_number"])
                                     else 0,
                    "Stage Label":   latest["stage"],
                    "Trend Score":   int(latest["trend_score"]),
                    "Confidence":    round(latest["stage_confidence"], 1),
                    "RS (3M)":       round(latest["rs"] * 100, 2)
                                     if not np.isnan(latest["rs"]) else np.nan,
                    "RS Percentile": round(latest["rs_percentile"], 1)
                                     if not np.isnan(latest["rs_percentile"])
                                     else np.nan,
                    "Transitioning": latest["stage_transitioning"],
                    "MA20":          round(latest["ma20"], 2),
                    "MA50":          round(latest["ma50"], 2),
                    "MA200":         round(latest["ma200"], 2),
                    "ATR%":          round(latest["atr_pct"] * 100, 2),
                    "Vol Ratio":     round(latest["volume_ratio"], 2),
                    "Accum Days":    int(
                                         classified["accumulation_day"]
                                         .tail(10).sum()
                                     ),
                    "Dist Days":     int(
                                         classified["distribution_day"]
                                         .tail(10).sum()
                                     ),
                })

                print(f"{latest['stage']} | Score: {int(latest['trend_score'])} | Conf: {round(latest['stage_confidence'], 1)}%")

            except Exception as e:
                print(f"ERROR — {e}")
                continue

        if not results:
            print("No results returned.")
            return pd.DataFrame()

        df_results = pd.DataFrame(results)

        # apply filters
        if min_score > 0:
            df_results = df_results[df_results["Trend Score"] >= min_score]

        if stage_filter is not None:
            df_results = df_results[df_results["Stage"] == stage_filter]

        # sort by trend score then confidence
        df_results = df_results.sort_values(
            ["Trend Score", "Confidence"],
            ascending=False
        ).reset_index(drop=True)

        return df_results

    # =========================================================
    # LATEST STAGE — SINGLE STOCK
    # =========================================================
    def latest_stage(self, df: pd.DataFrame) -> dict:

        latest = df.iloc[-1]

        return {
            "date":          latest.name,
            "close":         round(latest["Close"], 2),
            "stage":         latest["stage"],
            "stage_number":  latest["stage_number"],
            "trend_score":   int(latest["trend_score"]),
            "confidence":    round(latest["stage_confidence"], 1),
            "rs_3m":         round(latest["rs"] * 100, 2)
                             if not np.isnan(latest["rs"]) else None,
            "rs_percentile": round(latest["rs_percentile"], 1)
                             if not np.isnan(latest["rs_percentile"]) else None,
            "transitioning": latest["stage_transitioning"],
        }

    # =========================================================
    # MOVING AVERAGES — wider slope window
    # =========================================================
    def _moving_averages(self, df):

        c = self.config

        df["ma20"]  = df["Close"].rolling(c.short_ma).mean()
        df["ma50"]  = df["Close"].rolling(c.medium_ma).mean()
        df["ma200"] = df["Close"].rolling(c.long_ma).mean()

        for ma in ["ma20", "ma50", "ma200"]:
            # normalize slope as % per bar — comparable across price levels
            df[f"{ma}_slope"] = (
                (df[ma] - df[ma].shift(c.slope_period))
                / df[ma].shift(c.slope_period)
            ) / c.slope_period * 100

    # =========================================================
    # ATR
    # =========================================================
    def _atr(self, df):

        hl  = df["High"] - df["Low"]
        hc  = np.abs(df["High"] - df["Close"].shift(1))
        lc  = np.abs(df["Low"]  - df["Close"].shift(1))

        tr       = pd.concat([hl, hc, lc], axis=1).max(axis=1)
        df["ATR"]     = tr.rolling(self.config.atr_period).mean()
        df["atr_pct"] = df["ATR"] / df["Close"]

    # =========================================================
    # RELATIVE STRENGTH — 3 month + percentile rank
    # =========================================================
    def _relative_strength(self, df):

        if "benchmark_close" not in df.columns:
            df["benchmark_close"] = np.nan

        p = self.config.rs_period

        stock_ret     = df["Close"] / df["Close"].shift(p) - 1
        benchmark_ret = df["benchmark_close"] / df["benchmark_close"].shift(p) - 1

        df["rs"] = stock_ret - benchmark_ret

        # RS trend — is RS improving vs its own 20-bar average
        df["rs_trend"] = df["rs"] > df["rs"].rolling(20).mean()

        # RS percentile rank over 1 year — where does current RS sit historically
        df["rs_percentile"] = df["rs"].rolling(252).rank(pct=True) * 100

    # =========================================================
    # VOLUME — 50-bar average, institutional grade
    # =========================================================
    def _volume_analysis(self, df):

        df["avg_volume"]   = df["Volume"].rolling(self.config.volume_period).mean()
        df["volume_ratio"] = df["Volume"] / df["avg_volume"]

        # accumulation day — up on above-average volume
        df["accumulation_day"] = (
            (df["Close"] > df["Close"].shift(1))
            & (df["volume_ratio"] > 0.5)
        )

        # distribution day — down on above-average volume
        df["distribution_day"] = (
            (df["Close"] < df["Close"].shift(1))
            & (df["volume_ratio"] > 0.5)
        )

        # churning — high volume but little price progress (topping signal)
        df["churning"] = (
            (df["volume_ratio"] > 0.5)
            & (np.abs(df["Close"] - df["Close"].shift(1)) / df["Close"] < 0.005)
        )

    # =========================================================
    # TREND STRUCTURE
    # =========================================================
    def _trend_structure(self, df):

        lb = self.config.pivot_lookback

        df["rolling_high"] = df["High"].rolling(lb).max()
        df["rolling_low"]  = df["Low"].rolling(lb).min()

        df["higher_high"]  = df["rolling_high"] > df["rolling_high"].shift(lb)
        df["higher_low"]   = df["rolling_low"]  > df["rolling_low"].shift(lb)
        df["lower_high"]   = df["rolling_high"] < df["rolling_high"].shift(lb)
        df["lower_low"]    = df["rolling_low"]  < df["rolling_low"].shift(lb)

    # =========================================================
    # WEIGHTED TREND SCORE — not all signals equal
    # =========================================================
    def _trend_score(self, df):

        score = np.zeros(len(df))

        # price vs MAs — weight by importance
        score += (df["Close"] > df["ma20"]).astype(int)   * 1
        score += (df["Close"] > df["ma50"]).astype(int)   * 2   # heavier weight
        score += (df["Close"] > df["ma200"]).astype(int)  * 2   # heavier weight

        # full MA alignment — most important single condition
        score += (
            (df["ma20"] > df["ma50"]) & (df["ma50"] > df["ma200"])
        ).astype(int) * 2

        # positive slopes
        score += (df["ma20_slope"]  > 0).astype(int) * 1
        score += (df["ma50_slope"]  > 0).astype(int) * 1
        score += (df["ma200_slope"] > 0).astype(int) * 1        # NEW

        # swing structure
        score += df["higher_high"].astype(int) * 1
        score += df["higher_low"].astype(int)  * 1

        # RS improving AND above benchmark
        score += (
            df["rs_trend"].fillna(False)
            & (df["rs"].fillna(0) > 0)
        ).astype(int) * 1

        # volume confirmation — accumulation days in last 10 bars
        score += (
            df["accumulation_day"].rolling(10).sum() >= 3
        ).astype(int) * 1

        # churning penalty — topping signal
        score -= df["churning"].astype(int) * 1

        df["trend_score"] = score.clip(lower=0)

    # =========================================================
    # STAGE CLASSIFICATION — improved logic
    # =========================================================
    def _classify_stages(self, df):

        c = self.config

        # ── STAGE 2: MARKUP ──────────────────────────────────
        # Requires MA200 slope too — prevents classifying late-stage
        # rallies where 200 MA is still falling as Stage 2
        stage2 = (
            (df["trend_score"] >= c.stage2_min_score)
            & (df["Close"]    > df["ma50"])
            & (df["ma20"]     > df["ma50"])
            & (df["ma50"]     > df["ma200"])
            & (df["ma20_slope"]  > c.stage2_ma50_slope_min)
            & (df["ma50_slope"]  > c.stage2_ma50_slope_min)
            & (df["ma200_slope"] > c.stage2_ma200_slope_min)   # NEW
        )

        # ── STAGE 4: DECLINE ─────────────────────────────────
        # Added RS condition — must be underperforming benchmark
        stage4 = (
            (df["Close"]      < df["ma50"])
            & (df["ma20"]     < df["ma50"])
            & (df["ma50"]     < df["ma200"])
            & (df["ma20_slope"]  < 0)
            & (df["ma50_slope"]  < 0)
            & (df["lower_high"])
            & (df["lower_low"])
            & (df["rs"].fillna(0) < c.stage4_rs_penalty)       # NEW
        )

        # ── STAGE 1: ACCUMULATION ────────────────────────────
        # Tighter thresholds — real accumulation is tight and quiet
        stage1 = (
            (np.abs(df["ma20_slope"])  < c.stage1_slope_max)
            & (np.abs(df["ma50_slope"]) < c.stage1_slope_max)
            & (df["atr_pct"]            < c.stage1_atr_max)
            & (
                np.abs(
                    (df["Close"] - df["ma50"]) / df["ma50"]
                ) < c.stage1_ma50_proximity
            )
            & (~stage2)
            & (~stage4)
        )

        # ── STAGE 3: DISTRIBUTION ────────────────────────────
        # Everything not cleanly Stage 1/2/4
        stage3 = ~(stage1 | stage2 | stage4)

        df["stage"] = np.select(
            [stage1, stage2, stage3, stage4],
            [
                "Stage 1 - Accumulation",
                "Stage 2 - Markup",
                "Stage 3 - Distribution",
                "Stage 4 - Decline",
            ],
            default="Unknown"
        )

        df["stage_number"] = np.select(
            [stage1, stage2, stage3, stage4],
            [1, 2, 3, 4],
            default=np.nan
        )

    # =========================================================
    # STAGE CONFIDENCE — how strongly does price fit the stage
    # =========================================================
    def _stage_confidence(self, df):
      """
       Confidence = how strongly price fits its current stage.
       - Stage 2 (Long):    high score = high confidence
       - Stage 4 (Short):   low score  = high confidence
       - Stage 1/3:       proximity to midpoint 7 = high confidence
      """

      SCORE_MAX = 14.0
      SCORE_MIN = 0.0
      SCORE_MID = 7.0
      SHORT_CEILING = 6.0    # practical ceiling for Stage 4 stocks
      ts = df["trend_score"]


      # ── Stage 2 confidence — how close to historical peak
      long_confidence = (ts / SCORE_MAX * 100).clip(0, 100)

      # ── Stage 4 confidence — how close to historical trough
      # invert: low score = high confidence for shorts
      short_confidence = short_confidence = np.where(
                         ts <= SHORT_CEILING,
                         ((SHORT_CEILING - ts) / SHORT_CEILING * 100).clip(0, 100),
                         # above ceiling — still show some confidence but low
                         ((SCORE_MAX - ts) / SCORE_MAX * 100).clip(0, 100)
                        )

      # ── Stage 1/3 confidence — how close to median (sideways)
      neutral_confidence = ((1 - abs(ts - SCORE_MID) / SCORE_MID) * 100).clip(0, 100)

      # ── Apply correct confidence per stage
      df["stage_confidence"] = np.select(
        [
            df["stage_number"] == 2,
            df["stage_number"] == 4,
            df["stage_number"].isin([1, 3]),
        ],
        [
            long_confidence,
            short_confidence,
            neutral_confidence,
        ],
        default=50 ).clip(0, 100)

    # =========================================================
    # STAGE TRANSITIONS — detect when stage is changing
    # =========================================================
    def _stage_transitions(self, df):
        """
        Flags bars where stage has changed vs previous bar.
        Useful for catching early stage shifts.
        """

        df["stage_transitioning"] = (
            df["stage_number"] != df["stage_number"].shift(1)
        )

In [64]:
# =============================================================
# EXAMPLE — MULTI STOCK SCANNER
# =============================================================

def run_scanner(watchlist=None, stage_filter=None, start="2022-01-01"):

    if watchlist is None:
        print("No watchlist provided. Please pass a list of tickers.")
        return

    clf = BrianShannonAuctionClassifier()

    print(f"\n{'='*55}")
    print(f"  MULTI STOCK SCANNER — {len(watchlist)} tickers")
    print(f"{'='*55}")

    results = clf.scan(
        tickers=watchlist,
        benchmark="SPY",
        start=start,
        stage_filter=stage_filter
    )

    if results.empty:
        print("No stocks matched the filter.")
        return

    stage_label = f"Stage {stage_filter} Only" if stage_filter else "All Stages"

    print(f"\n{'='*55}")
    print(f"  SCAN RESULTS — {stage_label}")
    print(f"{'='*55}\n")

    display_cols = [
        "Ticker", "Close", "Stage Label", "Trend Score",
        "Confidence", "RS (3M)", "RS Percentile",
        "Transitioning", "Accum Days", "Dist Days"
    ]

    #print(results[display_cols].to_string(index=False))
    #print(f"\nTotal matches: {len(results)}")

    return results


In [65]:
# todays list
todays_list = etfs # declining_stocks['ETF'].tolist()
#run_single("NVDA")
# Multi stock scanner — Stage 4 only
results = run_scanner(watchlist=todays_list)

filtered = results[
  # ── Must be Stage 2 — confirmed uptrend
   (results["Stage"] >= 3)

  # ── Trend must be strong — not borderline
  & (results["Trend Score"] <= 4)

  # ── High confidence the stage call is correct
  & (results["Confidence"] >= 60)

  # ── RS must be negative — lagging the market
  #& (results["RS (3M)"] < 0)

  ].copy()

# normalize trend weakness to 0-100
# score of 0 = 100% weak = best short
# score of 4 = 0% weak = weakest short candidate
filtered["trend_weakness_normalized"] = (
    (14 - filtered["Trend Score"]) / 14 * 100
)


# ── Rank by composite score: RS Percentile + Confidence + Trend Score
filtered["rank_score"] = (
        # lower RS percentile = worse = better short
        (100 - filtered["RS Percentile"]) * 0         # secondary — how weak is RS historically

        # lower trend score = weaker = better short
        + filtered["trend_weakness_normalized"]  * 1   # primary — how broken is the trend

    )



filtered = filtered.sort_values(
        "rank_score", ascending=False
    ).head(1000).reset_index(drop=True)

display_cols = [
        "Ticker", "Close", "Stage Label", "Trend Score",
        "Confidence", "RS (3M)",
        "Transitioning"
    ]


df_o = df_o[df_o['Asset'].isin(filtered['Ticker'])]
etfs = df_o['Asset'].to_list()
print(etfs)
print(len(etfs))


  MULTI STOCK SCANNER — 2 tickers

[1/2] Processing SARO... Stage 4 - Decline | Score: 0 | Conf: 100.0%
[2/2] Processing FN... Stage 4 - Decline | Score: 0 | Conf: 100.0%

  SCAN RESULTS — All Stages

['SARO', 'FN']
2


In [17]:
def macdv(prices, fast=12, slow=26, signal=9, atr_window=10, thresholds=(50, 150)):
    """
    Compute MACD-V (volatility normalized MACD).

    Parameters
    ----------
    prices : pd.Series
        Price series (e.g. closing prices).
    fast : int
        Fast EMA period.
    slow : int
        Slow EMA period.
    signal : int
        Signal EMA period for MACD line.
    atr_window : int
        ATR lookback window.
    thresholds : tuple
        (lower, upper) thresholds for neutral/ranging and extreme momentum zones.

    Returns
    -------
    pd.DataFrame with columns:
        - MACDV : MACD-V value
        - Signal : EMA of MACDV
        - Histogram : MACDV - Signal
        - EntryFlag : True when momentum is strong enough, False otherwise
    """
    # --- Step 1: EMAs for MACD ---
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_raw = ema_fast - ema_slow

    # --- Step 2: ATR for normalization ---
    high = prices.shift(1) * (1 + 0.01)   # synthetic highs/lows if OHLC not available
    low = prices.shift(1) * (1 - 0.01)
    close = prices
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(atr_window).mean()

    # --- Step 3: Normalize MACD by ATR ---
    macdv = (macd_raw / atr) * 100

    # --- Step 4: Signal line and histogram ---
    signal_line = macdv.ewm(span=signal, adjust=False).mean()
    histogram = macdv - signal_line

    # --- Step 5: Entry conditions ---
    lower, upper = thresholds
    entry_flag = ((macdv.abs() > lower) & (macdv.abs() < upper)) | (macdv.abs() > upper)

    df = pd.DataFrame({
        "MACDV": macdv,
        "Signal": signal_line,
        "Histogram": histogram,
        "EntryFlag": entry_flag
    })
    curr = df.iloc[-1]

    return curr.EntryFlag

def money_flow_signals(df, period=10):
    """
    Calculate Money Flow Index (MFI) and generate signals:
    - Positive money flow (TP > TP_prev)
    - Divergence (Price vs MFI mismatch)

    Parameters:
        df (pd.DataFrame): DataFrame with columns ["High", "Low", "Close", "Volume"]
        period (int): Lookback period for MFI (default=10)

    Returns:
        pd.DataFrame with added columns: ["TypicalPrice", "MFI", "PositiveFlow", "Divergence"]
    """

    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Step 1: Typical Price
    df["TypicalPrice"] = (df["High"] + df["Low"] + df["Close"]) / 3

    # Step 2: Raw Money Flow
    df["RawMoneyFlow"] = df["TypicalPrice"] * df["Volume"]

    # Step 3: Positive & Negative Flow
    df["PositiveFlow"] = np.where(df["TypicalPrice"] > df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)
    df["NegativeFlow"] = np.where(df["TypicalPrice"] < df["TypicalPrice"].shift(1), df["RawMoneyFlow"], 0.0)

    # Step 4: Money Flow Ratio & MFI
    pos_flow = df["PositiveFlow"].rolling(period).sum()
    neg_flow = df["NegativeFlow"].rolling(period).sum()
    money_flow_ratio = pos_flow / neg_flow.replace(0, np.nan)
    df["MFI"] = 100 - (100 / (1 + money_flow_ratio))

    # Step 5: Positive Money Flow Signal
    df["PositiveFlowSignal"] = df["TypicalPrice"] > df["TypicalPrice"].shift(1)

    # Step 6: Divergence Detection
    df["PriceHigh"] = df["Close"].rolling(period).max()
    df["PriceLow"] = df["Close"].rolling(period).min()
    df["MFIHigh"] = df["MFI"].rolling(period).max()
    df["MFILow"] = df["MFI"].rolling(period).min()

    def divergence(row):
        if np.isnan(row["MFI"]):
            return None
        # Price makes higher high, but MFI does not
        if row["Close"] >= row["PriceHigh"] and row["MFI"] < row["MFIHigh"]:
            return "Bearish Divergence ⚠️"
        # Price makes lower low, but MFI does not
        elif row["Close"] <= row["PriceLow"] and row["MFI"] > row["MFILow"]:
            return "Bullish Divergence ✅"
        else:
            return "No Divergence"

    df["Divergence"] = df.apply(divergence, axis=1)
    df['Entry_signal']= (df["MFI"] > 50) & (df["Divergence"] == "No Divergence")
    curr = df.iloc[-1]

    return curr.Entry_signal

def anchored_vwap_old(ticker, anchor_date):
    """
    Calculate Anchored VWAP starting from a given anchor_date.
    Works with both single-level and multi-level columns (e.g. yfinance output).
    """
    # --- Step 1: Flatten columns if multi-index ---
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # --- Step 2: Ensure required columns exist ---
    required_cols = ["High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Step 3: Subset from anchor_date ---
    df_anchor = df.loc[df.index >= pd.to_datetime(anchor_date)].copy()
    if df_anchor.empty:
        raise ValueError(f"No data found on/after {anchor_date}")

    # --- Step 4: Compute typical price ---
    df_anchor["typical_price"] = (df_anchor["High"] + df_anchor["Low"] + df_anchor["Close"]) / 3.0

    # --- Step 5: Cumulative PV and VWAP ---
    df_anchor["cum_pv"] = (df_anchor["typical_price"].astype(float) * df_anchor["Volume"].astype(float)).cumsum()
    df_anchor["cum_vol"] = df_anchor["Volume"].astype(float).cumsum()
    df_anchor["anchored_vwap"] = df_anchor["cum_pv"] / df_anchor["cum_vol"]

    return df_anchor[["anchored_vwap"]]


In [18]:
# Filter ETFs or stocks for liquidity
def filter_by_liquidity(etf_df, ticker_col="Asset", min_dollar_vol=1e6, lookback_days=30):
    liquid_etfs = []

    for ticker in etf_df[ticker_col]:
        try:
            # Fetch daily historical data
            data = yf.download(ticker, period=f"{lookback_days*2}d", interval="1d", auto_adjust=True)

            if data.empty:
                continue

            # Calculate dollar volume (Close × Volume)
            data["dollar_volume"] = data["Close"] * data["Volume"]

            # Calculate rolling average over lookback_days
            avg_dollar_volume = data["dollar_volume"].rolling(window=lookback_days).mean().iloc[-1]

            # Check liquidity condition
            if avg_dollar_volume >= min_dollar_vol:
                liquid_etfs.append(ticker)

        except Exception as e:
            print(f"Error fetching {ticker}: {e}")

    # Return filtered DataFrame
    return etf_df[etf_df[ticker_col].isin(liquid_etfs)]

# Example usage
df = pd.DataFrame({"Assets": etfs})
liquid_df = filter_by_liquidity(df, ticker_col="Assets")
df_o = df_raw[df_raw['Asset'].isin(liquid_df['Assets'])]
etfs2 = df_o['Asset'].to_list()
etfs_clean = list(dict.fromkeys(etfs2))


print("")
print(etfs_clean)
print(len(etfs_clean))



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


['SARO', 'FN']
2


In [39]:

def anchored_vwap_structural(
    ticker: str,
    lookback_weeks: int = 5,
    pivot_left: int = 2,
    pivot_right: int = 2):
    """
    Anchors VWAP from the last STRUCTURAL swing low
    that led to a Lower High (LH), within a lookback window.
    """

    try:
        # ----------------------------
        # 1. Download data
        # ----------------------------
        data = yf.download(
            ticker,
            period="3mo",
            interval="1d",
            auto_adjust=True,
            progress=False
        )

        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        lookback_days = lookback_weeks * 5
        if len(data) < lookback_days:
            raise ValueError("Not enough data")

        recent = data.tail(lookback_days)

        # ----------------------------
        # 2. Find STRUCTURAL swing lows
        # ----------------------------
        swing_lows = []
        for i in range(pivot_left, len(recent) - pivot_right):
            window = recent['Low'].iloc[i - pivot_left : i + pivot_right + 1]
            if recent['Low'].iloc[i] == window.min():
                swing_lows.append(recent.index[i])

        if not swing_lows:
            raise ValueError("No swing lows found")

        # ----------------------------
        # 3. Find LL that caused a LH
        # ----------------------------
        anchor_date = None

        for sl in reversed(swing_lows):
            after_sl = recent.loc[sl:]

            highs = after_sl['High']
            for i in range(1, len(highs)):
                # LH definition: failed attempt to make HH
                if highs.iloc[i] < highs.iloc[i - 1]:
                    anchor_date = sl
                    break

            if anchor_date is not None:
                break

        if anchor_date is None:
            # No structural breakdown
            data['Anchored_VWAP'] = np.nan
            data['Signal'] = False
            return data[['Anchored_VWAP', 'Signal']]

        # ----------------------------
        # 4. Anchor VWAP from STRUCTURAL LL
        # ----------------------------
        anchor_data = data.loc[anchor_date:]

        typical_price = (
            anchor_data['High']
            + anchor_data['Low']
            + anchor_data['Close']
        ) / 3

        volume = anchor_data['Volume']

        pv = (typical_price * volume).cumsum()
        v = volume.cumsum()

        avwap = pv / v.where(v != 0, np.nan)

        data['Anchored_VWAP'] = np.nan
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap

        # ----------------------------
        # 5. Final signal logic
        # ----------------------------
        latest_close = data['Close'].iloc[-1]
        swing_low_price = data.loc[anchor_date, 'Low']

        data['Signal'] = (
            (data['Close'] < data['Anchored_VWAP']) &
            (latest_close < swing_low_price) &
            (data['Anchored_VWAP'].notna())
        )

        print(
            f"{ticker} | AVWAP anchored from {anchor_date.date()} "
            f"(structural LL @ {swing_low_price:.2f})"
        )

        return data[['Anchored_VWAP', 'Signal']]

    except Exception as e:
        print(f"{ticker} error: {e}")
        return None


# Function to fetch historical weekly data
def rolling_regression_slope(series, window=10):
    """Rolling linear regression slope (price units per bar)."""
    def calc_slope(y):
        if len(y) < 2:
            return np.nan
        x = np.arange(len(y))
        slope, _, _, _, _ = linregress(x, y)
        return slope
    return series.rolling(window).apply(calc_slope, raw=False)


def anchored_vwap_old(ticker, anchor_date):
    """
    Calculate Anchored VWAP starting from a given anchor_date.
    Works with both single-level and multi-level columns (e.g. yfinance output).
    """
    # --- Step 1: Flatten columns if multi-index ---
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # --- Step 2: Ensure required columns exist ---
    required_cols = ["High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # --- Step 3: Subset from anchor_date ---
    df_anchor = df.loc[df.index >= pd.to_datetime(anchor_date)].copy()
    if df_anchor.empty:
        raise ValueError(f"No data found on/after {anchor_date}")

    # --- Step 4: Compute typical price ---
    df_anchor["typical_price"] = (df_anchor["High"] + df_anchor["Low"] + df_anchor["Close"]) / 3.0

    # --- Step 5: Cumulative PV and VWAP ---
    df_anchor["cum_pv"] = (df_anchor["typical_price"].astype(float) * df_anchor["Volume"].astype(float)).cumsum()
    df_anchor["cum_vol"] = df_anchor["Volume"].astype(float).cumsum()
    df_anchor["anchored_vwap"] = df_anchor["cum_pv"] / df_anchor["cum_vol"]

    return df_anchor[["anchored_vwap"]]

def anchored_vwap(ticker: str, lookback_weeks: int = 5):
    """
    Calculate the Anchored VWAP from the most recent high within the past `lookback_weeks`
    and create a 'Signal' column that gives 'Buy' when Close > Anchored_VWAP, else 'No-Buy'.

    Args:
        ticker (str): Stock ticker symbol
        lookback_weeks (int): Number of weeks to look back for the highest price

    Returns:
        pandas.DataFrame: DataFrame with OHLC, Volume, Anchored_VWAP, and Signal columns
    """
    try:
        # --- Fetch 6 months of daily data to cover the lookback window
        data = yf.download(ticker, period="3mo", interval="1d", progress=False,auto_adjust=True)
        if isinstance(data.columns, pd.MultiIndex):
          data.columns = data.columns.get_level_values(0)  # keep only first level

        if data.empty:
            raise ValueError(f"No data retrieved for ticker {ticker}")

        data.dropna(inplace=True)
        data.index = pd.to_datetime(data.index)

        # --- Ensure we have enough data for the lookback period
        min_days_required = lookback_weeks * 5  # ~5 trading days per week
        if len(data) < min_days_required:
            raise ValueError(f"Insufficient data: {len(data)} days available, {min_days_required} required")

        # --- Find the most recent high within the past lookback_weeks
        recent_period = data.tail(min_days_required)
        anchor_date = recent_period['Low'].idxmin()

        # --- Verify anchor_date is a valid Timestamp
        if not isinstance(anchor_date, pd.Timestamp):
            raise ValueError(f"Invalid anchor_date: {anchor_date}. Expected a Timestamp.")

        #anchor_price = recent_period.loc[anchor_date, 'Low']
        anchor_price = recent_period['Low'].min()

        # --- Use .date() safely since we confirmed anchor_date is a Timestamp
        print(f"Anchored VWAP for {ticker} starting from {anchor_date.date()} (recent low = {anchor_price:.2f})")

        # --- Slice data from the anchor date onwards
        anchor_data = data.loc[anchor_date:]

        # --- Compute VWAP starting from the anchor date
        # Use typical price ((H+L+C)/3) for more accurate VWAP
        typical_price = (anchor_data['High'] + anchor_data['Low'] + anchor_data['Close']) / 3
        q = anchor_data['Volume']
        pv = (typical_price * q).cumsum()
        v = q.cumsum()

        # --- Avoid division by zero
        avwap = pv / v.where(v != 0, np.nan)

        # --- Add Anchored VWAP to the full dataset
        data['Anchored_VWAP'] = np.nan  # Initialize with NaN
        data.loc[anchor_date:, 'Anchored_VWAP'] = avwap

        # --- Create Buy/No-Buy Signal
        # Only apply signal where Anchored_VWAP is not NaN
        data['Signal'] = np.where(
            (data['Close'] < data['Anchored_VWAP']) & (data['Anchored_VWAP'].notna()),
            True,
            False
        )

        return data[['Anchored_VWAP', 'Signal']]

    except Exception as e:
        print(f"Error processing {ticker}: {str(e)}")
        return None

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1mo",auto_adjust=True)


      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      df['slope_raw'] = rolling_regression_slope(df['10_month_SMA'], window=5)
      # normalize as % change per month — comparable across all price levels
      df["slope_pct"] = df["slope_raw"] / df["10_month_SMA"] * 100
      # bearish threshold — SMA rising more than 1% per month
      df["slope_bearish"] = df["slope_pct"] <- 1.0
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="5y", interval="1wk",auto_adjust=True)


      df['10_week_SMA'] = df['Close'].rolling(window=10).mean()
      df['30_week_SMA'] = df['Close'].rolling(window=30).mean()
      df['10_EMA'] = df['Close'].ewm(span=10, adjust=False).mean()
      df['20_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
      # slope on 10 week SMA — short term trend direction
      df["slope_10sma"] = rolling_regression_slope(df["10_week_SMA"], window=10)
      # slope on 30 week SMA — long term trend direction
      df["slope_30sma"] = rolling_regression_slope(df["30_week_SMA"], window=10)
      # normalize both as % per week — comparable across stocks
      df["slope_10sma_pct"] = df["slope_10sma"] / df["10_week_SMA"] * 100
      df["slope_30sma_pct"] = df["slope_30sma"] / df["30_week_SMA"] * 100

      # 10 week SMA rising at least 0.5% per week
      df["slope_10_bearish"] = df["slope_10sma_pct"] < -0.5
      # 30 week SMA rising at least 0.3% per week
      df["slope_30_bearish"] = df["slope_30sma_pct"] < -0.75
      # both rising = confirmed bearish trend
      df["both_slopes_bearish"] = (
              df["slope_10_bearish"] & df["slope_30_bearish"])

      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      # --- Overhead Resistance Filter ---
      recent_52_weeks = df[-52:]
      min_close_52w = recent_52_weeks['Close'].min()
      last_close = df['Close'].iloc[-1].iloc[0]
      # --- Above 52 weeks Low ---
      df['above_52w_low'] = last_close > min_close_52w
      df['below_52w_low'] = last_close < min_close_52w

      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(df):
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] < curr['Signal_Line'].iloc[-1]
      below_zero_line = curr['MACD_Line'].iloc[-1] < 0


      return macd_crossover, below_zero_line

    except Exception as e:
      print("Something went wrong while computing the MACD", e)

def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None


def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)


# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)


# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    latest_price = df['Close'].iloc[-1]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    price_ema = df['8_day_EMA'].iloc[-1]
    atr_multiple = 1.55 # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr
    #stop      = price_ema + (trailing * atr_multiple)

    return trailing, price_ema
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['50_day_avg_volume'] = df['Volume'].rolling(window=50).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['15_day_EMA'] = df['Close'].ewm(span=15, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['5_day_SMA'] = df['Close'].rolling(window=5).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df['trend_break'] = df['Close'] < df['Close'].rolling(5).min().shift(1)
    df['swing_high'] = df['High'].where(df['trend_break']).ffill()

    # Slope for 5-day SMA (short-term trend)
    df['slope5_raw'] = rolling_regression_slope(df['5_day_SMA'], window=10)
    df['slope5_pct_per_day'] = df['slope5_raw'] / df['5_day_SMA']  # fractional change per day
    df['slope5_annualized_pct'] = df['slope5_pct_per_day'] * 252 * 100  # % per year
    # Slope for 50-day SMA (intermediate-term trend)
    df['slope50_raw'] = rolling_regression_slope(df['50_day_SMA'], window=10)
    df['slope50_pct_per_day'] = df['slope50_raw'] / df['50_day_SMA']  # fractional change per day
    df['slope50_annualized_pct'] = df['slope50_pct_per_day'] * 252 * 100  # % per year
    df['100_day_SMA'] = df['Close'].rolling(window=100).mean()
    df['200_day_SMA'] = df['Close'].rolling(window=200).mean()
    df['ATR'] = compute_atr(df, 10)
    df["upper_bound"] = df["8_day_EMA"] + 1.5* df["ATR"]
    df["lower_bound"] = df["8_day_EMA"] - 0.5* df["ATR"]
    # 1️⃣ Yesterday touched or pierced 8 EMA
    df['prior_touch_8ema'] = df['Low'].shift(1) <= df['8_day_EMA'].shift(1)
    # 2️⃣ Today closes above 8 EMA
    df['close_above_8ema'] = df['Close'] > df['8_day_EMA']
    # 3️⃣ Today closes above yesterday’s close (price rising)
    df['close_above_prev_close'] = df['Close'] > df['Close'].shift(1)
    # 4️⃣ Strong confirmation: Break previous high
    df['break_prev_high'] = df['Close'] > df['High'].shift(1)
    # 5️⃣ 8 EMA slope positive (trend filter)
    df['ema8_rising'] = df['8_day_EMA'] > df['8_day_EMA'].shift(1)
    # EMA 8 slope
    df['ema8_slope'] = df['8_day_EMA'] - df['8_day_EMA'].shift(1)
    # EMA 8 slope previous
    df['ema8_slope_prev'] = df['ema8_slope'].shift(1)
    # EMA 8 acceleration
    df['ema8_accel'] = df['ema8_slope'] - df['ema8_slope_prev']
    # EMA 8 slope direction (1 = up, 0 = down)
    df['ema8_dir'] = (df['ema8_slope'] > 0).astype(int)
    # Count direction changes over last 5 days
    df['ema8_direction_changes'] = (
      df['ema8_dir']
      .diff()
      .abs()
      .rolling(5)
      .sum()
      )
    # from here
    # 2. Avoid strong counter-trend moves and chop
    df['daily_return'] = df['Close'].pct_change()
    avoid_strong_up = df['daily_return'] > 0.035          # Strong bullish day
    avoid_chop = df['ema8_direction_changes'] >= 3
    # 3. EMA 8 Price Action
    df['touched_ema8']      = df['High'] >= df['8_day_EMA'] * 0.995
    df['closed_below_ema8'] = df['Close'] < df['8_day_EMA']
    df['bearish_candle']    = df['Close'] < df['Open']
    df['lower_low'] = df['Low'] < df['Low'].shift(1) * 0.995  # 0.5% lower

    # Strong upper wick rejection
    df['upper_wick_ratio']  = (df['High'] - df['Close']) / (df['High'] - df['Low'] + 0.0001)
    df['strong_upper_wick'] = df['upper_wick_ratio'] > 0.60

    # 4. Advanced Bearish Patterns
    df['bearish_engulfing'] = (
      (df['Close'] < df['Open']) &
      (df['Open'] > df['Close'].shift(1)) &
      (df['Close'] < df['Close'].shift(1))
    )

    # Intraday failed break above (single candle)
    df['failed_break_above_intraday'] = (
       (df['High'] > df['8_day_EMA']) &      # today's high pierced EMA
       (df['Close'] < df['8_day_EMA'])        # but closed back below
    )

    # Two-day version
    df['failed_break_above_twoday'] = (
       (df['Close'].shift(1) > df['8_day_EMA'].shift(1)) &
       (df['Close'] < df['8_day_EMA'])
    )

    df['failed_break_ema8'] = (
      df['failed_break_above_intraday'] |
      df['failed_break_above_twoday']
    )

    df['near_50sma'] = df['High'] >= df['50_day_SMA'] * 0.99
    df['weak_close'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'] + 1e-6) < 0.4

    # 5. A and A+ Setups
    df['A_setup'] = (
      df['touched_ema8'] &
      df['closed_below_ema8'] &
      df['bearish_candle'] &
      df['strong_upper_wick']
      #df['weak_close']
    )

    df['A_plus_setup'] = (
      df['failed_break_ema8'] &
      #(df['bearish_engulfing'] | df['strong_upper_wick']) &
      df['bearish_engulfing'] &
      df['closed_below_ema8'] &
      (df['near_50sma'] | df['strong_upper_wick'])
    )

    # Trend Continuation / Momentum Trades ---
    # NOTE: Entry requires price to be within 1 ATR of 8 EMA (handled upstream)
    df['trend_continuation'] = (
      (df['Close'] < df['8_day_EMA']) &                     # Below EMA
      (df['Close'].shift(1) < df['8_day_EMA'].shift(1)) &   # Was already below
      #(df['High'].shift(1) >= df['8_day_EMA'].shift(1)) &  # Optional: rejection wick
      df['lower_low'] &                                     # Making lower lows
      (df['ema8_slope'] < 0)                                # EMA sloping down
      & (df['daily_return'] < -0.005)                       # strong bearish momentum
    )

    # 6. Entry Trigger (Momentum)
    df['entry_trigger'] = df['Low'] < df['Low'].shift(1) * 0.995  # 0.5% lower
    # --- Trend ---
    trend_short = df['slope50_raw'] < 0

    # ====================== FINAL SHORT SIGNAL ======================
    df['short_signal'] = (
        trend_short &
        (~avoid_strong_up) &
        (~avoid_chop) &
        (
          df['A_setup'] |
          df['A_plus_setup'] |
          (df['trend_continuation'] & df['entry_trigger'])  # entry trigger only for continuation
        )
    )


    # Signal Strength Labeling
    df['signal_type'] = 'C+'
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & df['trend_continuation'], 'signal_type'] = 'Hybrid'
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & ~df['trend_continuation'], 'signal_type'] = 'Pullback (A/A+)'
    df.loc[df['short_signal'] & df['trend_continuation'] & ~(df['A_setup'] | df['A_plus_setup']), 'signal_type'] = 'Trend Continuation'

    df['signal_strength'] = 'C+'
    # Pure A+
    df.loc[df['short_signal'] & df['A_plus_setup'] & ~df['trend_continuation'],'signal_strength' ] = 'A+'
    # Then A
    df.loc[df['short_signal'] & df['A_setup'] & ~df['A_plus_setup'], 'signal_strength'] = 'A'
    # Then B+ (only if NOT A or A+)
    df.loc[df['short_signal'] & df['trend_continuation'] & ~(df['A_setup'] | df['A_plus_setup']),'signal_strength'] = 'B+'
    # Hybrid — both pullback and continuation aligning
    df.loc[df['short_signal'] & (df['A_setup'] | df['A_plus_setup']) & df['trend_continuation'], 'signal_strength'] = 'A+'

    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
    # Compute raw EFI
    df['EFI'] = (df['Close'].diff()) * df['Volume']
    # Compute EMA of EFI
    df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
    # Determine if EFI_EMA is rising or falling
    df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
    # Calculate ADX, +DMI and -DMI
    high = df['High']
    low = df['Low']
    close = df['Close']
    # Calculate directional movements
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    # Calculate True Range (TR)
    tr1 = high - low
    tr2 = (high - close.shift()).abs()
    tr3 = (low - close.shift()).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    # Smooth TR, +DM, and -DM using Wilder’s smoothing
    atr = tr.rolling(window=10).sum()
    plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
    minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)
    plus_dm_smoothed = plus_dm_series.rolling(window=10).sum()
    minus_dm_smoothed = minus_dm_series.rolling(window=10).sum()
    # Directional Indicators
    plus_di = 100 * (plus_dm_smoothed / atr)
    minus_di = 100 * (minus_dm_smoothed / atr)
    # DX and ADX
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=10).mean()
    # Add results to original DataFrame
    df['+DI'] = plus_di
    df['-DI'] = minus_di
    df['ADX'] = adx
    df['di_flag'] = df['-DI'] > df['+DI']
    df['di_flag'] = df['di_flag'].astype(int)
    df['adx_indicator'] = np.where(df['ADX'] > 20, 1, 0)
    df['adx_signal'] = df['adx_indicator'] * df['di_flag']
    return df

# Function to fetch hourly data
def get_30mins_data(ticker):
    df = yf.download(ticker, interval='30m', period='60d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level

    df['65d_SMA'] = df['Close'].rolling(window=65).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['65d_SMA'], window=10)
    # Normalize → fractional change per 30-minute bar
    df['slope_norm'] = df['slope_raw'] / df['65d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 13
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    # Short-term & medium-term moving averages
    df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['50_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['200_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']

    return df


# Function to fetch hourly data
def get_15min_data(ticker):
    df = yf.download(ticker, interval='15m', period='30d',auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)  # keep only first level
    df['130d_SMA'] = df['Close'].rolling(window=130).mean()
    # Slope calculation: regression over recent 10 bars (~5 trading hours)
    df['slope_raw'] = rolling_regression_slope(df['130d_SMA'], window=10)
    # Normalize → fractional change per 15-minute bar
    df['slope_norm'] = df['slope_raw'] / df['130d_SMA']
    # Annualize to % per year (standard for 30m regular hours)
    bars_per_year = 252 * 26
    df['SMA_Slope'] = df['slope_norm'] * bars_per_year * 100
    return df

def get_heikin_ashi_signal(ticker="AAPL", period="6mo", interval="1d"):
    # Fetch OHLC data
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # keep only first level

    # Compute Heikin Ashi candles
    ha_df = pd.DataFrame(index=df.index)
    ha_df['HA_Close'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

    ha_open = []
    for i in range(len(df)):
        if i == 0:
             ha_open.append((df['Open'].iloc[i] + df['Close'].iloc[i]) / 2)
        else:
            ha_open.append((ha_open[i-1] + ha_df['HA_Close'].iloc[i-1]) / 2)
    ha_df['HA_Open'] = ha_open
    #ha_df['HA_High'] = ha_df[['HA_Open', 'HA_Close']].assign(High=df['High']).max(axis=1)
    #ha_df['HA_Low'] = ha_df[['HA_Open', 'HA_Close']].assign(Low=df['Low']).min(axis=1)
    ha_df['HA_High'] = pd.concat([ha_df['HA_Open'], ha_df['HA_Close'], df['High']], axis=1).max(axis=1)
    ha_df['HA_Low'] = pd.concat([ha_df['HA_Open'], ha_df['HA_Close'], df['Low']], axis=1).min(axis=1)

    # Combine with original
    df = df.join(ha_df)
    # Check for green candle with flat bottom
    last = df.iloc[-1]
    red_candle = last['HA_Close'] < last['HA_Open']
    flat_top = abs(last['HA_High'] - last['HA_Open']) < 0.001 * last['HA_Open']
    signal = red_candle and flat_top

    print(f"\n🔍 Checking {ticker} ({interval} timeframe)")
    print(f"HA_Open: {last['HA_Open']:.2f}, HA_Close: {last['HA_Close']:.2f}, HA_Low: {last['HA_Low']:.2f}")
    if signal:
        print("✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!")
    elif red_candle:
        print("🟢 Candle is red but not flat-topped — still bearish, but less strong.")
    else:
        print("🔴 Not a bearish candle — no entry confirmation yet.")

    return signal, red_candle

# Function to check monthly trend
def is_monthly_trend_bearish(df):
    if df.empty:
        return False


    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['10_month_SMA'].iloc[-1]
    sma_slope  = df["slope_bearish"].iloc[-1]
    macd_bearish_signal = df['MACD_Line'].iloc[-1] < df['Signal_Line'].iloc[-1]
    below_10_month_SMA = (latest_price < latest_sma)
    return below_10_month_SMA


# Function to check weekly trend
def is_weekly_trend_bearish(df2):
    if df2.empty:
        return False

    df = df2.copy()
    latest_price  = df['Close'].iloc[-1].iloc[0]
    latest_10sma  = df['10_week_SMA'].iloc[-1]
    below_10w_SMA = latest_price < latest_10sma
    latest_30sma  = df['30_week_SMA'].iloc[-1]
    below_30w_SMA = latest_price < latest_30sma
    sma_slope     = df["slope_30_bearish"].iloc[-1]
    macd_bearish_signal = df['MACD_Line'].iloc[-1] < df['Signal_Line'].iloc[-1]
    trend_ok      = below_10w_SMA and below_30w_SMA
    #elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Falling'
    #elderforce_ema_ok = df['EFI_EMA'].iloc[-1] < 0

    time.sleep(2)  # Add a delay of 1 second between requests

    return  trend_ok


# Function to check daily entry signal
def is_daily_entry_bearish(df2):
    if df2.empty:
        return False


    df = df2.copy()

    counter_trend_short_signal = df['short_signal'].iloc[-1]
    latest_price = df['Close'].iloc[-1] #.iloc[0]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    latest_100sma = df['100_day_SMA'].iloc[-1]
    latest_200sma = df['200_day_SMA'].iloc[-1]
    below_50sma = latest_price < latest_50sma
    below_100sma = latest_price < latest_100sma
    below_200sma = latest_price < latest_200sma
    is_50sma_below_100sma = latest_50sma < latest_100sma
    is_50sma_below_200sma = latest_50sma < latest_200sma
    is_100sma_below_200sma = latest_100sma < latest_200sma
    sma_slope_5  = df['slope5_annualized_pct'].iloc[-1] < -10
    sma_slope_50 = df['slope50_annualized_pct'].iloc[-1] < 0
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Falling'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] < 0
    macd_bearish_signal, below_zero_line = is_macd_bullish(df)
    adx_ok = df['adx_signal'].iloc[-1] == 1
    slopes_ok =  sma_slope_5 and sma_slope_50
    moving_averages_ok = below_50sma and below_100sma and below_200sma \
                         and (is_100sma_below_200sma or is_50sma_below_200sma or is_50sma_below_100sma )


    # Look for a breakout above 20-day SMA & RSI > 50
    return moving_averages_ok and sma_slope_50  #adx_ok and sma_slope_50 #and elderforce_ema_ok

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price        = df['Close'].iloc[-1]
      latest_sma          = df['50_day_SMA'].iloc[-1]
      latest_price_8ema   =  df['8_day_EMA'].iloc[-1]
      upper_bound         = df['upper_bound'].iloc[-1]
      lower_bound         = df['lower_bound'].iloc[-1]
      atr                  = df['ATR'].iloc[-1]
      signal_strength      = df['signal_strength'].iloc[-1]
      # Define tight A-Line band
      aline_lower          = latest_price_8ema

      mfi_signal          = money_flow_signals(df)
      # Print results
      print(f"\nMoney Outflow indicator for {ticker} is:")
      print(not(mfi_signal))

      df_entry             = get_30mins_data(ticker)
      latest_priceh        = df_entry['Close'].iloc[-1]
      latest_priceh_5sma   = df_entry['65d_SMA'].iloc[-1]
      slope_hr             = df_entry['SMA_Slope'].iloc[-1] < 0
      priceh_buy           = latest_priceh < latest_priceh_5sma
      HA_sell_signal_h,rc_h= get_heikin_ashi_signal(ticker, period="90d", interval="1d")

      df_refined_entry     = get_15min_data(ticker)
      latest_pricem        = df_refined_entry['Close'].iloc[-1]
      latest_pricem_5sma   = df_refined_entry['130d_SMA'].iloc[-1]
      pricem_buy           = latest_pricem < latest_pricem_5sma
      slope_m              = df_refined_entry['SMA_Slope'].iloc[-1] < 0

      # Strict entry — flat top red HA required
      refined_entry_signal_strict = (
          slope_hr and
          priceh_buy and
          HA_sell_signal_h )

      # Standard entry — just red HA candle required
      refined_entry_signal_standard = (
          slope_hr and
          priceh_buy and
          rc_h   )

      # Use strict for A+ signals, standard for A/B+
      if signal_strength == 'A+' :
        refined_entry_signal = refined_entry_signal_standard or True
      else:
        refined_entry_signal = refined_entry_signal_standard or True

      if latest_price >  upper_bound:
        entry_signal = "Extended Short Entry"  ## > 1.1 ATR (too stretched)
      elif latest_price >= latest_price_8ema  :
          if refined_entry_signal:
             entry_signal = "True Trend Short Entry"
          else:
             entry_signal = "Skip"
      elif latest_price >= lower_bound and (rc_h or HA_sell_signal_h ):
        entry_signal = "Aline Short Entry"
      else:
        entry_signal = "Skip"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)


        if is_monthly_trend_bearish(monthly_df):
            if  is_weekly_trend_bearish(weekly_df) and is_daily_entry_bearish(daily_df):
                entry_signal = "Bearish Entry Confirmed ✅"
            else:
                entry_signal = "No Bearish Entry Yet on Lower Timeframes ⏳"
        else:
            entry_signal = "Monthly Trend is not Bearish ❌"

        results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results



In [30]:
# Multi-time frame entry Check
etfs_to_check = etfs_clean

df_signals = check_mtf_entry(etfs_to_check)


df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]

df_final.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,Asset,Entry_Signal
0,SARO,Bearish Entry Confirmed ✅
1,FN,Bearish Entry Confirmed ✅


## Generate Sell list

In [40]:
df_final = df_signals[df_signals['Entry_Signal'] =="Bearish Entry Confirmed ✅"]
final_etfs_to_check = df_final['Asset'].tolist()
#final_etfs_to_check = etfs_clean
to_remove = ["PX"]
final_etfs_to_check = [x for x in final_etfs_to_check if x not in to_remove]
sell_list = check_entry_conditions(final_etfs_to_check)


sell_list= sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry', 'True Trend Short Entry',"Extended Short Entry" ])]

sell_list.head()

[*********************100%***********************]  1 of 1 completed



Money Outflow indicator for SARO is:
True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking SARO (1d timeframe)
HA_Open: 24.61, HA_Close: 24.32, HA_Low: 24.03
✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



Money Outflow indicator for FN is:
True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



🔍 Checking FN (1d timeframe)
HA_Open: 416.66, HA_Close: 396.36, HA_Low: 391.34
✅ Heikin Ashi candle is Red with a FLAT top — strong bearish signal!


[*********************100%***********************]  1 of 1 completed


,Asset,Entry_Signal


# Find and filter correlated assets to reduce concentration risk.

In [ ]:
def check_abnormal_buying(df, lookback=10, volume_threshold=2.5,
                          price_threshold=0.02):
    """
    Checks last lookback bars for abnormal buying activity.
    Flags if any bar shows both volume surge AND strong positive close.
    Used as a short trade safety check — abnormal buying = avoid or exit.

    Parameters
    ----------
    df                 : DataFrame with Close, Volume columns
    lookback           : bars to check for recent activity
    volume_threshold   : z-score above which volume is abnormal (default 2.5)
    price_threshold    : minimum positive return to flag (default 2%)

    Returns
    -------
    dict with detection flag, recency, risk level and description
    """

    df = df.copy()

    # ensure daily_return exists
    df["daily_return"] = df["Close"].pct_change()

    # baseline stats from a stable lookback window
    # use bars -60 to -10 to avoid including recent abnormal activity
    baseline    = df["Volume"].iloc[-60:-10]
    vol_mean    = baseline.mean()
    vol_std     = baseline.std()

    if vol_std == 0 or np.isnan(vol_std):
        return {
            "abnormal_buying_detected": False,
            "days_since":               None,
            "high_risk":                False,
            "risk_level":               "Unknown",
            "description":              "Insufficient volume history for z-score calculation"
        }

    recent = df.iloc[-lookback:].copy()

    # volume z-score relative to baseline
    recent["volume_zscore"] = (recent["Volume"] - vol_mean) / vol_std

    # abnormal buying = volume spike AND strong positive close
    recent["abnormal_buying"] = (
        (recent["volume_zscore"] > volume_threshold)
        & (recent["daily_return"] > price_threshold)
    )

    abnormal_detected = recent["abnormal_buying"].any()

    if abnormal_detected:
        # find how many bars ago the MOST RECENT flag occurred
        reversed_flags = recent["abnormal_buying"].values[::-1]
        days_since     = int(reversed_flags.argmax()) + 1

        # peak volume z-score on flagged days
        peak_zscore = round(
            recent.loc[recent["abnormal_buying"], "volume_zscore"].max(), 2
        )

        # peak return on flagged days
        peak_return = round(
            recent.loc[recent["abnormal_buying"], "daily_return"].max() * 100, 2
        )

        # recency determines risk level
        if days_since <= 2:
            risk_level  = "EXTREME"
            description = (
                f"Abnormal buying {days_since} bar(s) ago — "
                f"volume {peak_zscore}x std devs above baseline, "
                f"price up {peak_return}%. DO NOT SHORT."
            )
        elif days_since <= 5:
            risk_level  = "HIGH"
            description = (
                f"Abnormal buying {days_since} bars ago — "
                f"volume {peak_zscore}x std devs, price up {peak_return}%. "
                f"Avoid new short entries."
            )
        else:
            risk_level  = "MODERATE"
            description = (
                f"Abnormal buying detected {days_since} bars ago — "
                f"fading but monitor closely before shorting."
            )

        high_risk = days_since <= 3

    else:
        days_since  = None
        high_risk   = False
        risk_level  = "LOW"
        description = "No abnormal buying detected — short entry not flagged by volume."

    return {
        "abnormal_buying_detected": abnormal_detected,
        "days_since":               days_since,
        "high_risk":                high_risk,
        "risk_level":               risk_level,
        "description":              description,
    }



def get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66, period="3mo", interval="1d"):
    """
    Filters a ranked list of tickers to return only uncorrelated picks.

    Parameters:
    -----------
    tickers : list
        All candidate tickers.
    ranked_picks : list
        Ranked list of tickers (best to worst).
    threshold : float
        Correlation threshold (default 0.66).
    period : str
        Data period for yfinance (default "3mo").
    interval : str
        Data interval (default "1d").

    Returns:
    --------
    final_selection : list
        List of uncorrelated tickers.
    corr_matrix : DataFrame
        Correlation matrix of daily returns.
    """
    # Step 1: Get prices
    data = yf.download(tickers, period=period, interval=interval,auto_adjust=True)["Close"]
    data = data.ffill()

    # Step 2: Convert to daily returns
    returns = data.pct_change().dropna()

    # Step 3: Correlation matrix
    corr_matrix = returns.corr()

    # Step 4: Filter uncorrelated picks
    final_selection = []
    for pick in ranked_picks:
        if all(abs(corr_matrix.loc[pick, sel]) <= threshold for sel in final_selection):
            final_selection.append(pick)

    return final_selection, corr_matrix



In [ ]:
# Apply TA filters and prioritize ETFs
results = []
sell_list = sell_list[sell_list['Entry_Signal'].isin(['Aline Short Entry','True Trend Short Entry'])]
quad_witching_date = get_last_quad_witching()  # auto-detects most recent

for etf in sell_list['Asset'].to_list():
   df           = get_daily_data(etf)
   buying_check = check_abnormal_buying(df)
   abnormal_buying_check = buying_check['high_risk']
   ab_days_ago = buying_check['days_since']
   ab_risk_level = buying_check['risk_level']
   ab_description = buying_check['description']
   print(f"\nFor your ticker  :", etf)
   print(f"\nAbnormal buying check is :", abnormal_buying_check)
   print(f"\nAbnormal buying occured (days ago)  :", ab_days_ago)
   print(f"\nAbnormal buying risk level is :", ab_risk_level)
   print(f"\nAbnormal buying description is :", ab_description)
   price        = df['Close'].iloc[-1]
   swing_high   = df['swing_high'].iloc[-1]
   below_50sma  = price  < df['50_day_SMA'].iloc[-1]
   sma_slope_50 = df['slope50_annualized_pct'].iloc[-1]< -25
   sma_slope_5 = df['slope5_annualized_pct'].iloc[-1]< 0
   vwap_df2     = anchored_vwap_old(etf, start_of_year)
   ytd_vwap     = vwap_df2['anchored_vwap'].iloc[-1]
   below_ytd_vwap = price < ytd_vwap
   vwap_qw     = anchored_vwap_old(etf, quad_witching_date)
   qw_vwap     = vwap_qw['anchored_vwap'].iloc[-1]
   below_qw_vwap = price < qw_vwap
   print("Current price is :", price)
   print("Most recent quad witching AVWAP is :", qw_vwap)
   print("Year to date VWAP is :", ytd_vwap)
   signal_strength = df['signal_strength'].iloc[-1]
   print("Signal Strength is :", signal_strength)
   signal_filter = (signal_strength == 'A+' or signal_strength == 'A' or signal_strength == 'B+')

   signal_type = df['signal_type'].iloc[-1]
   print("Signal Type is :", signal_type)

   #vwap_df     = anchored_vwap(etf, lookback_weeks=4)
   vwap_df     = anchored_vwap_structural(etf)
   vwap        = vwap_df['Anchored_VWAP'].iloc[-1]
   vwap_signal = vwap_df['Signal'].iloc[-1]
   below_vwap  = price < vwap
   print("Anchored VWAP from correction swing high is :", vwap)
   # MTD
   #vwap_mtd     = anchored_vwap_old(etf, first_day_month)
   #mtd_vwap    = vwap_mtd['anchored_vwap'].iloc[-1]
   #below_mtd_vwap = price < mtd_vwap
   #print("MTD VWAP is :", mtd_vwap)
   atr_multiple_map = {
     'A+': 1.55,
     'A':  1.55,
     'B+': 2.0,
     'C+': 2.0 }
   atr_multiple = atr_multiple_map.get(signal_strength, 1.55)
   # Only use swing high if it's within last 10 bars, otherwise fall back to ATR stop
   swing_high_age = df['trend_break'] .iloc[-10:].any()


   if (sma_slope_5 and sma_slope_50 and below_vwap and below_qw_vwap and not abnormal_buying_check and below_ytd_vwap):
    trail, price_ema = calculate_risk_reward(df)
    #trail = calculate_risk_reward(df)
    entry_price = price - min(0.25, 0.05*trail)
    stop_loss = price_ema + (atr_multiple*trail)
    if swing_high_age:
      stop = np.maximum(stop_loss, swing_high + 1*trail)
    else:
      stop = stop_loss # fall back to pure ATR stop
    risk = np.abs(stop - entry_price)
    breakeven_trigger = entry_price - (1.0 * risk)
    take_profit = entry_price - (2*risk)
    reward =  entry_price - take_profit
    support_level = stop
    risk_reward_ratio = reward / risk
    # Ensure risk is greater than zero before division
    if risk > 0:

        rr_ratio  = reward / risk
    else:
        rr_ratio = np.nan

    stop_loss_perc = ((entry_price-support_level )/entry_price )*100
    take_profit_perc = (( entry_price - take_profit)/entry_price )*100
    # Fetch the Entry_Signal from buy_list
    entry_signal = sell_list.loc[sell_list['Asset'] == etf, 'Entry_Signal'].values[0]

    # Append results with Entry_Signal
    results.append({
            "Asset": etf,
            "Risk-Reward": rr_ratio,
            "breakeven": breakeven_trigger,
            "Stop Loss": support_level,
            "Take Profit": take_profit,
            #"Current Price": price,
            "Entry Price": entry_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "stop_loss_perc": stop_loss_perc,
            "take_profit_perc": take_profit_perc,
            "Anchored VWAP": vwap,
            #"MTD VWAP": mtd_vwap
            "signal_type": signal_type,
            "signal_strength": signal_strength

        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=True).reset_index(drop = True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  df_results = pd.DataFrame({"Asset": ["No Asset available"]})

df2 = df_results.merge(df_o[['Asset','Type', 'score']], on='Asset', how='left')
df2['timestamp'] = datetime.now()
df2 = df2.sort_values(by='score', ascending=True)
df2.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


For your ticker  : DLB

Abnormal buying check is : False

Abnormal buying occured (days ago)  : None

Abnormal buying risk level is : LOW

Abnormal buying description is : No abnormal buying detected — short entry not flagged by volume.



[*********************100%***********************]  1 of 1 completed


Current price is : 49.15999984741211
Most recent quad witching AVWAP is : 50.89670732871136
Year to date VWAP is : 61.694000908561165
Signal Strength is : C+
Signal Type is : C+
DLB | AVWAP anchored from 2026-07-20 (structural LL @ 48.55)
Anchored VWAP from correction swing high is : 49.61526701788979


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


For your ticker  : GMED

Abnormal buying check is : False

Abnormal buying occured (days ago)  : None

Abnormal buying risk level is : LOW

Abnormal buying description is : No abnormal buying detected — short entry not flagged by volume.
Current price is : 75.41999816894531
Most recent quad witching AVWAP is : 79.0722740666404
Year to date VWAP is : 78.37860813808429
Signal Strength is : C+
Signal Type is : C+


GMED | AVWAP anchored from 2026-07-21 (structural LL @ 74.51)
Anchored VWAP from correction swing high is : 75.52949823413181


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


For your ticker  : LOPE

Abnormal buying check is : False

Abnormal buying occured (days ago)  : None

Abnormal buying risk level is : LOW

Abnormal buying description is : No abnormal buying detected — short entry not flagged by volume.
Current price is : 139.2100067138672
Most recent quad witching AVWAP is : 144.9795620871863
Year to date VWAP is : 169.76767308309223
Signal Strength is : C+
Signal Type is : C+


LOPE | AVWAP anchored from 2026-07-15 (structural LL @ 138.34)
Anchored VWAP from correction swing high is : 140.5875305964459


## Sentiment Score

In [ ]:
# =============================================================
# CATALYST + SENTIMENT ENGINE — FREE TIER ONLY
# =============================================================

class CatalystSentimentEngine:
    """
    Uses only Finnhub free tier endpoints:
    - company_news()          ← headlines + catalyst detection
    - recommendation_trends() ← analyst sentiment proxy
    """

    def __init__(self, api_key: str):
        self.client  = finnhub.Client(api_key=api_key)
        self.results = []

    # =========================================================
    # SENTIMENT FROM ANALYST RECOMMENDATIONS — FREE
    # =========================================================

    def _recommendation_score(self, rec: dict) -> float:

      strong_buy  = rec.get("strongBuy", 0)
      buy         = rec.get("buy", 0)
      hold        = rec.get("hold", 0)
      sell        = rec.get("sell", 0)
      strong_sell = rec.get("strongSell", 0)

      total = (
        strong_buy + buy +
        hold +
        sell + strong_sell
      )

      if total == 0:
          return 0

      score = (
        strong_buy * 2 +
        buy * 1 +
        hold * 0 +
        sell * -1 +
        strong_sell * -2 ) / (total * 2)

      return score

    def _sentiment_from_recommendations(self, ticker: str) -> tuple:
        """
        Uses analyst recommendation trends as sentiment proxy.
        Returns (sentiment_label, score, detail)
        """
        try:
            recs = self.client.recommendation_trends(ticker)

            if not recs:
                return "NEUTRAL", 0.0, "No analyst data"

            # most recent period
            #latest = recs[0]
            latest = recs[0]

            latest_score = self._recommendation_score(latest)

            if len(recs) > 1:

              previous = recs[1]
              previous_score = self._recommendation_score(previous)
              rec_momentum = latest_score - previous_score

            else:
              rec_momentum = 0

            if rec_momentum > 0.10:
               momentum_label = "UPGRADE TREND"
            elif rec_momentum < -0.10:
               momentum_label = "DOWNGRADE TREND"
            else:
              momentum_label = "STABLE"

            strong_buy  = latest.get("strongBuy",  0)
            buy         = latest.get("buy",         0)
            hold        = latest.get("hold",        0)
            sell        = latest.get("sell",        0)
            strong_sell = latest.get("strongSell",  0)

            total = strong_buy + buy + hold + sell + strong_sell

            if total == 0:
                return "NEUTRAL", 0.0, "No recommendations"

            # weighted score — ranges from -1 (all strong sell) to +1 (all strong buy)
            score = (
                (strong_buy * 2 + buy * 1 + hold * 0
                 + sell * -1 + strong_sell * -2)
                / (total * 2)
            )

            detail = (
                f"SB:{strong_buy} B:{buy} "
                f"H:{hold} S:{sell} SS:{strong_sell}"
                f"| {momentum_label} "
                f"({rec_momentum:.2f})"
            )

            if score >= 0.50:
                return "BUY", round(score, 3), detail, round(rec_momentum, 3)
            elif score >= 0.20:
                return "WEAK BUY", round(score, 3), detail, round(rec_momentum, 3)
            elif score <= -0.50:
                return "SELL", round(score, 3), detail, round(rec_momentum, 3)
            elif score <= -0.20:
                return "WEAK SELL", round(score, 3), detail, round(rec_momentum, 3)
            else:
                return "NEUTRAL", round(score, 3), detail, round(rec_momentum, 3)

        except Exception:
            return "NEUTRAL", 0.0, "Recommendation fetch failed",0.0

    # =========================================================
    # SENTIMENT FROM HEADLINE KEYWORDS — FREE
    # =========================================================
    def _sentiment_from_headlines(self, headlines: list) -> tuple:
        """
        Score sentiment directly from headline keywords.
        Returns (sentiment_label, score)
        """

        positive_words = [
            "beat", "record", "surge", "jump", "soar", "rally",
            "gain", "rise", "growth", "profit", "upgrade", "approval",
            "win", "awarded", "raised guidance", "strong", "exceed",
            "outperform", "bullish", "breakthrough", "launched",
            "partnership", "dividend increase", "buyback"
        ]

        negative_words = [
            "miss", "fall", "drop", "decline", "loss", "cut",
            "warning", "downgrade", "reject", "fail", "weak",
            "below", "concern", "risk", "lawsuit", "fine",
            "suspended", "cancelled", "resign", "fraud",
            "bankruptcy", "default", "sell", "bearish", "plunge"
        ]

        combined = " ".join(headlines).lower()

        pos_count = sum(1 for w in positive_words if w in combined)
        neg_count = sum(1 for w in negative_words if w in combined)
        total     = pos_count + neg_count

        if total == 0:
            return "NEUTRAL", 0.0

        score = (pos_count - neg_count) / total

        if score >= 0.40:
            return "BUY", round(score, 3)
        elif score >= 0.10:
            return "WEAK BUY", round(score, 3)
        elif score <= -0.40:
            return "SELL", round(score, 3)
        elif score <= -0.10:
            return "WEAK SELL", round(score, 3)
        else:
            return "NEUTRAL", round(score, 3)

    # =========================================================
    # CATALYST DETECTION
    # =========================================================
    def _detect_catalyst(self, headlines: list) -> str:

        catalyst_map = [
            (["earnings beat", "beat estimate", "beat expectations",
              "exceeded forecast", "record profit", "record earnings",
              "surpassed"], "Earnings Beat"),

            (["earnings miss", "missed estimate", "missed expectations",
              "below forecast", "profit warning",
              "earnings warning"], "Earnings Miss"),

            (["earnings", "quarterly result", "q1", "q2", "q3", "q4",
              "annual result", "full year", "half year"], "Earnings Report"),

            (["revenue beat", "revenue growth", "sales growth",
              "record revenue", "revenue surge"], "Revenue Growth"),

            (["guidance raised", "raised guidance", "raised outlook",
              "upgraded guidance", "raised forecast"], "Guidance Raised"),

            (["guidance cut", "lowered guidance", "profit warning",
              "revenue warning", "lowered outlook"], "Guidance Cut"),

            (["dividend increase", "dividend raised", "special dividend",
              "dividend hike"], "Dividend Increase"),

            (["dividend cut", "dividend suspended",
              "suspended dividend"], "Dividend Cut"),

            (["buyback", "share repurchase",
              "repurchase program"], "Buyback Announced"),

            (["merger", "acquisition", "takeover", "buyout",
              "acquired by", "deal agreed"], "M&A Activity"),

            (["upgrade", "upgraded to buy", "upgraded to outperform",
              "price target raised", "target raised"], "Analyst Upgrade"),

            (["downgrade", "downgraded to sell", "price target cut",
              "target lowered"], "Analyst Downgrade"),

            (["fda approval", "drug approval", "approved by fda",
              "regulatory approval"], "Regulatory Approval"),

            (["fda rejection", "clinical trial fail",
              "drug failed"], "Regulatory Rejection"),

            (["product launch", "new product", "launched",
              "unveiled"], "Product Launch"),

            (["contract win", "won contract", "awarded contract",
              "secured deal"], "Contract Win"),

            (["contract loss", "lost contract",
              "deal cancelled"], "Contract Loss"),

            (["partnership", "joint venture",
              "strategic alliance"], "Partnership"),

            (["ceo appointed", "new ceo",
              "chief executive appointed"], "New CEO"),

            (["ceo resigned", "ceo departure",
              "ceo steps down"], "CEO Departure"),

            (["lawsuit", "legal action", "sued",
              "regulatory fine", "penalty"], "Legal Risk"),

            (["bankruptcy", "chapter 11", "insolvency",
              "debt restructuring"], "Bankruptcy Risk"),

            (["short seller", "short report", "fraud allegation",
              "accounting irregularity"], "Short Attack"),

            (["interest rate", "fed decision",
              "rate hike", "rate cut"], "Macro — Rates"),

            (["sanctions", "tariff",
              "trade war", "export ban"], "Geopolitical Risk"),
        ]

        combined = " ".join(headlines).lower()

        for keywords, label in catalyst_map:
            if any(kw in combined for kw in keywords):
                return label

        return "General News"

    # =========================================================
    # ANALYSE ONE TICKER
    # =========================================================
    def analyse(self, ticker: str, days_back: int = 7) -> dict:

        today     = datetime.today()
        from_date = (today - timedelta(days=days_back)).strftime("%Y-%m-%d")
        to_date   = today.strftime("%Y-%m-%d")

        result = {
            "Ticker":       ticker,
            "Sentiment":    "NEUTRAL",
            "Catalyst":     "No News",
            "Signal":       "NEUTRAL — No News",
            "Score":        0.0,
            "News Count":   0,
            "Rec Momentum": 0.0,
            "Analyst Rec":  "—",
            "Top Headline": "—",
            "Date":         to_date,
        }

        try:
            # ── 1. Analyst Recommendations (free)
            rec_sentiment, rec_score, rec_detail, rec_momentum  = \
                self._sentiment_from_recommendations(ticker)

            result["Rec Momentum"] = rec_momentum
            result["Analyst Rec"] = rec_detail

            # ── 2. Company News (free)
            news = self.client.company_news(
                ticker,
                _from=from_date,
                to=to_date
            )

            if news:
                headlines = [
                    item.get("headline", "").lower()
                    for item in news[:20]
                ]

                result["News Count"]   = len(news)
                result["Top Headline"] = news[0].get(
                    "headline", "—"
                )[:100]
                result["Catalyst"]     = self._detect_catalyst(headlines)

                # headline sentiment
                news_sentiment, news_score = \
                    self._sentiment_from_headlines(headlines)

                # ── 3. Combine — analyst rec + headline sentiment
                # weight: analyst 60%, headline 40%
                #combined_score = (rec_score * 0.60) + (news_score * 0.40)
                combined_score = (
                   rec_score * 0.50 +
                   news_score * 0.35 +
                   rec_momentum * 0.15 )

                if combined_score >= 0.35:
                    final_sentiment = "BUY"
                elif combined_score >= 0.10:
                    final_sentiment = "WEAK BUY"
                elif combined_score <= -0.35:
                    final_sentiment = "SELL"
                elif combined_score <= -0.10:
                    final_sentiment = "WEAK SELL"
                else:
                    final_sentiment = "NEUTRAL"

            else:
                # no news — fall back to analyst rec only
                final_sentiment  = rec_sentiment
                combined_score   = rec_score

            result["Sentiment"] = final_sentiment
            result["Score"]     = round(combined_score, 3)
            result["Signal"]    = (
                f"{final_sentiment} — {result['Catalyst']}"
            )

        except Exception as e:
            result["Signal"]   = f"ERROR — {str(e)[:60]}"
            result["Catalyst"] = "Error"

        return result

    # =========================================================
    # BATCH SCAN
    # =========================================================
    def scan(
        self,
        tickers:   list,
        days_back: int   = 7,
        delay:     float = 1.0
    ) -> pd.DataFrame:

        print(f"\n{'='*60}")
        print(f"  CATALYST + SENTIMENT SCAN — {len(tickers)} tickers")
        print(f"  Looking back {days_back} days")
        print(f"  Using: Company News + Analyst Recommendations")
        print(f"{'='*60}\n")

        results = []

        for i, ticker in enumerate(tickers, 1):

            print(f"[{i:>3}/{len(tickers)}] {ticker:<10}", end=" ")

            result = self.analyse(ticker, days_back)
            results.append(result)

            print(f"{result['Signal']}")

            time.sleep(delay)

        df = pd.DataFrame(results)

        sentiment_order = {
            "BUY":       1,
            "WEAK BUY":  2,
            "NEUTRAL":   3,
            "WEAK SELL": 4,
            "SELL":      5,
        }

        df["_sort"] = df["Sentiment"].map(
            sentiment_order
        ).fillna(3)

        df = df.sort_values("_sort").drop(
            columns=["_sort"]
        ).reset_index(drop=True)

        return df

    # =========================================================
    # DISPLAY
    # =========================================================
    def display(self, df: pd.DataFrame):

        if df.empty:
            print("No results.")
            return

        sections = {
            "✅  BUY SIGNALS":      ["BUY", "WEAK BUY"],
            "❌  SELL SIGNALS":     ["SELL", "WEAK SELL"],
            "⬜  NEUTRAL":          ["NEUTRAL"],
        }

        for header, sentiments in sections.items():
            subset = df[df["Sentiment"].isin(sentiments)]
            if subset.empty:
                continue
            print(f"\n  ── {header}")
            for _, row in subset.iterrows():
                print(
                    f"     {row['Ticker']:<8} "
                    f"{row['Signal']:<45} "
                    f"Score: {row['Score']:>6.3f}  |  "
                    f"Mom: {row['Rec Momentum']:>6.3f}  |  "
                    f"News: {row['News Count']}"
                )

        print(f"\n  ── TOP HEADLINES")
        for _, row in df.iterrows():
            if row["Top Headline"] != "—":
                print(
                    f"     {row['Ticker']:<8} "
                    f"{row['Top Headline'][:75]}"
                )

    def save(
        self,
        df:   pd.DataFrame,
        path: str = "catalyst_sentiment.csv"
    ):
        df.to_csv(path, index=False)
        print(f"\nSaved → {path}")


# =============================================================
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    #tickers = ["NVDA", "AAPL", "META", "MSFT", "AMZN", "TSLA", "GOOGL", "JPM", "V", "AVGO"]
    FINNHUB_API_KEY = "d8iqq71r01qtcvnt9lmgd8iqq71r01qtcvnt9ln0"   #
    tickers = df2['Asset'].tolist()

    engine  = CatalystSentimentEngine(api_key=FINNHUB_API_KEY)
    results = engine.scan(tickers, days_back=7)

    engine.display(results)
    engine.save(results)

    # filter — earnings catalyst only
    #earnings = results[
        #results["Catalyst"].str.contains("Earnings")
      #]
    earnings = results.copy()
    if not earnings.empty:
        print(f"\n{'='*60}")
        print(f"  EARNINGS CATALYST STOCKS")
        print(f"{'='*60}")
        print(
            earnings[[
                "Ticker", "Signal", "Score",
                "Rec Momentum", "Top Headline"
            ]].to_string(index=False)
        )

#earnings
# Merge selected columns from earnings into df2
columns_to_add = [
    "Sentiment",
    "Catalyst",
    #"Rec Momentum",
    "Analyst Rec"
]

df2 = df2.merge(
    earnings[["Ticker"] + columns_to_add],
    left_on="Asset",
    right_on="Ticker",
    how="left"          # Keeps all rows from df2
)

# Optional: Drop the duplicate Ticker column (since you already have Asset)
df2 = df2.drop(columns=["Ticker"])

# Reorder columns nicely (optional but recommended)
#desired_order = ["Asset"] + columns_to_add + [col for col in df2.columns
                                              #if col not in ["Asset"] + columns_to_add]

#df2 = df2[desired_order]

print("✅ Merge completed successfully!")
print(f"Shape of df2 after merge: {df2.shape}")
print("\nNew columns added:")

df2

## Seasonality

In [ ]:

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    yfinance sometimes returns MultiIndex columns even for a single
    ticker depending on version / auto_adjust settings. This collapses
    them back to a flat Close/Open/High/Low/Volume frame so downstream
    .iloc[] lookups return scalars instead of Series.
    """
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = df.columns.get_level_values(0)
    return df


def get_seasonality_metrics(
    tickers: list,
    benchmark: str = 'SPY',
    holding_period: int = 20,
    lookback_years: int = 10,
    min_valid_years: int = 3
):
    """
    Analyze historical seasonality for multiple tickers vs benchmark.

    Returns:
        - Detailed DataFrame with all periods
        - Summary ranking DataFrame
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    today = datetime.today().date()
    start_date = f"{today.year - lookback_years - 1}-01-01"

    print(f"Downloading benchmark {benchmark}...")
    bench = yf.download(benchmark, start=start_date, progress=False, auto_adjust=True)
    if bench.empty:
        raise ValueError(f"Could not download benchmark {benchmark}")

    bench = _flatten_columns(bench)

    # force a plain 1-D Series regardless of how yfinance shaped it
    bench_close = bench['Close']
    if isinstance(bench_close, pd.DataFrame):
        bench_close = bench_close.iloc[:, 0]
    bench_close = bench_close.dropna()

    results = []
    summary_list = []

    print(f"Analyzing {len(tickers)} tickers...\n")

    for ticker in tickers:
        print(f"Processing {ticker}...", end=" ")

        try:
            stock = yf.download(ticker, start=start_date, progress=False, auto_adjust=True)
            if stock.empty or len(stock) < holding_period * 2:
                print("→ Insufficient data")
                continue

            stock = _flatten_columns(stock)

            stock_close = stock['Close']
            if isinstance(stock_close, pd.DataFrame):
                stock_close = stock_close.iloc[:, 0]
            stock_close = stock_close.dropna()

            n_stock = len(stock_close)
            n_bench = len(bench_close)

            excess_returns = []
            beat_count = 0
            valid_count = 0

            for year in range(today.year - lookback_years, today.year):
                # Handle anniversary date (including Feb 29)
                try:
                    target_date = pd.Timestamp(year=year, month=today.month, day=today.day)
                except ValueError:
                    target_date = pd.Timestamp(year=year, month=today.month, day=28)

                # Find nearest trading day on or after target date
                stock_idx = int(stock_close.index.searchsorted(target_date, side='left'))
                bench_idx = int(bench_close.index.searchsorted(target_date, side='left'))

                # bounds check BEFORE any .iloc access — avoids IndexError
                # and avoids ever building an ambiguous Series comparison
                if stock_idx >= n_stock or bench_idx >= n_bench:
                    continue
                if (stock_idx + holding_period >= n_stock or
                        bench_idx + holding_period >= n_bench):
                    continue

                # .iloc on a guaranteed 1-D Series now returns a scalar
                entry_stock = float(stock_close.iloc[stock_idx])
                exit_stock = float(stock_close.iloc[stock_idx + holding_period])

                entry_bench = float(bench_close.iloc[bench_idx])
                exit_bench = float(bench_close.iloc[bench_idx + holding_period])

                if entry_stock == 0 or entry_bench == 0:
                    continue

                stock_ret = (exit_stock / entry_stock - 1) * 100
                bench_ret = (exit_bench / entry_bench - 1) * 0
                excess = stock_ret - bench_ret   # now a plain float

                if np.isnan(excess):
                    continue

                excess_returns.append(excess)
                if excess > 0:                    # scalar comparison — safe
                    beat_count += 1
                valid_count += 1

            if valid_count < min_valid_years:
                print(f"→ Only {valid_count} valid periods")
                continue

            excess_array = np.array(excess_returns)
            win_rate = beat_count / valid_count * 100

            summary_list.append({
                'Ticker': ticker,
                'ValidYears_WinRate': str(valid_count) + ' - ' + str(round(win_rate, 1)),
                #'Win_Rate_%': round(win_rate, 1),
                'Avg_Excess_%': round(float(np.mean(excess_array)), 2)
                #signal_type + ' - ' + signal_strength
                #'Median_Excess_%': round(float(np.median(excess_array)), 2),
                #'Std_Excess_%': round(float(np.std(excess_array)), 2),
                #'Min_Excess_%': round(float(np.min(excess_array)), 2),
                #'Max_Excess_%': round(float(np.max(excess_array)), 2),
            })

            for i, excess in enumerate(excess_returns):
                results.append({
                    'Ticker': ticker,
                    'Year': today.year - lookback_years + i,
                    'Excess_%': round(excess, 2)
                })

            print(f"→ Done ({valid_count} years)")

        except Exception as e:
            print(f"→ Error: {e}")
            continue

    if not summary_list:
        print("No valid data found for any ticker.")
        return pd.DataFrame(), pd.DataFrame()

    summary_df = pd.DataFrame(summary_list)
    summary_df = summary_df.sort_values(by='Avg_Excess_%', ascending=False).reset_index(drop=True)

    detailed_df = pd.DataFrame(results)

    print("\n" + "=" * 100)
    print("SEASONALITY RANKING (vs Benchmark)")
    print("=" * 100)
    #print(summary_df.round(2).to_string(index=True))

    return detailed_df, summary_df



In [ ]:
tickers_list = df2['Asset'].tolist()

detailed, summary = get_seasonality_metrics(
        tickers=tickers_list,
        benchmark="SPY",
        holding_period=20,
        lookback_years=10,
        min_valid_years=3)

df2 = df2.merge(
    summary,
    left_on="Asset",
    right_on="Ticker",
    how="left"
)

# Drop duplicate Ticker column
if "Ticker" in df2.columns:
    df2 = df2.drop(columns=["Ticker"])

print("\n✅ News Sentiment merged into df2 successfully!")
print(f"Final shape: {df2.shape}")

df2.head()

## Shorter-Term Trend Quality

In [ ]:
# short
"""
One Month Quality Score — SHORT SIDE
======================================

Mirrors the long-side OneMonthQualityScore architecture, but is NOT
a simple sign-flip of every component. Several elements require
genuinely different logic for shorts, consistent with the earlier
mathematical edge analysis in this framework establishing that
shorts carry materially different (and generally worse) tail risk
than longs from gaps, squeezes, and volatility:

1. trend_score        -> Inverted: rewards CONSISTENT DOWNTREND
                          quality (negative slope, high R^2/t-stat),
                          same statistical foundation as the long
                          version, just scored on decline not advance.

2. structure_score     -> Inverted: rewards LOWER HIGHS + LOWER LOWS
                          via confirmed pivots, not HH/HL.

3. bounce_score        -> NOT a simple inversion of pullback_score.
                          For longs, pullback_score rewards SHALLOW
                          dips within an uptrend (low-risk entries).
                          For shorts, the equivalent low-risk entry
                          is a SHALLOW, ORDERLY BOUNCE within a
                          downtrend — entering shorts after a
                          dead-cat bounce fails is the short-side
                          analogue of buying a long pullback.
                          A short with ZERO bounces at all (straight
                          down) is actually higher squeeze-risk on
                          the next bounce, not higher quality.

4. breakdown_score      -> Inverted breakout_score: rewards genuine
                          breaks BELOW prior support (using only
                          prior bars, same look-ahead fix as the
                          long version), not breaks above resistance.

5. distribution_score   -> NOT accumulation_score inverted into
                          "selling score" naively. Measures whether
                          DOWN-days carry disproportionate volume
                          vs up-days (genuine institutional
                          distribution), same statistical approach
                          as volume_score but for the sell side.

6. volatility_penalty   -> Same annualized/percentile mechanics as
                          the long version, but with a STRUCTURALLY
                          HARSHER penalty weight. This reflects the
                          earlier finding in this framework that
                          elevated volatility is more dangerous on
                          shorts (gap risk, squeeze risk, unlimited
                          loss potential) than on longs.

7. NEW: squeeze_risk_penalty -> Genuinely new component with no long
                          equivalent. Penalizes short candidates
                          with characteristics that make them
                          vulnerable to a short squeeze independent
                          of trend quality — thin recent volume
                          relative to the move, and a high frequency
                          of sharp single-bar reversals within the
                          window.
"""

from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd
from scipy.signal import argrelextrema

try:
    import yfinance as yf
    _YFINANCE_AVAILABLE = True
except ImportError:
    _YFINANCE_AVAILABLE = False


# =============================================================
# CONFIG
# =============================================================

@dataclass
class ShortQualityScoreConfig:
    lookback:              int   = 20      # trading days (~1 month)
    pivot_order:           int   = 2        # bars each side for swing confirmation
    min_valid_bars:        int   = 15       # minimum bars required to score at all
    vol_lookback_long:     int   = 252       # 1y reference window for vol percentile
    distribution_vol_mult: float = 1.25     # volume threshold vs avg for "distribution day"

    # weights — must sum to 1.0 across the five POSITIVE components
    w_trend:        float = 0.26
    w_structure:    float = 0.20
    w_bounce:       float = 0.18
    w_breakdown:    float = 0.12
    w_distribution: float = 0.24   # weighted higher than long's w_volume —
                                     # confirmed institutional selling matters
                                     # more for short conviction than the
                                     # volume confirmation does for longs,
                                     # since shorts need stronger confirmation
                                     # given the asymmetric risk profile

    # subtractive penalties — both scaled 0-1, applied independently
    w_vol_penalty:     float = 0.20   # HARSHER than long's 0.15 —
                                        # reflects shorts' worse tail risk
                                        # from gaps/squeezes under volatility
    w_squeeze_penalty: float = 0.15   # NEW — no long-side equivalent

    def __post_init__(self):
        total = (self.w_trend + self.w_structure + self.w_bounce
                 + self.w_breakdown + self.w_distribution)
        if not np.isclose(total, 1.0, atol=1e-6):
            raise ValueError(
                f"Positive component weights must sum to 1.0, got {total:.4f}"
            )


# =============================================================
# CORE CLASS
# =============================================================

class OneMonthShortQualityScore:
    """
    Institutional-grade 1-month quality score for a SHORT system.

    Requires a DataFrame with at minimum:
        'close', 'high', 'low', 'volume'
    indexed chronologically (oldest -> newest).

    For statistically meaningful volatility percentile ranking,
    pass a longer history via `full_history` (e.g. 1y+) separate
    from the scoring window itself.

    A HIGH score here means a HIGH QUALITY short candidate —
    not a high quality stock. Scoring logic is built around
    confirming sustained institutional distribution within a
    genuine downtrend, with explicit penalties for the
    squeeze/gap risk that makes shorts asymmetrically more
    dangerous than longs when conviction is wrong.
    """

    def __init__(self, config: Optional[ShortQualityScoreConfig] = None):
        self.config = config or ShortQualityScoreConfig()

    # =========================================================
    # PUBLIC ENTRY POINT
    # =========================================================
    def calculate(
        self,
        df: pd.DataFrame,
        full_history: Optional[pd.DataFrame] = None
    ) -> dict:

        c = self.config

        required_cols = {"close", "high", "low", "volume"}
        missing = required_cols - set(df.columns.str.lower())
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        df = df.rename(columns=str.lower)
        window = df.tail(c.lookback).copy()

        if len(window) < c.min_valid_bars:
            return self._empty_result(
                reason=f"Insufficient bars: {len(window)} < {c.min_valid_bars}"
            )

        if window[["close", "high", "low", "volume"]].isna().any().any():
            window = window.dropna(subset=["close", "high", "low", "volume"])
            if len(window) < c.min_valid_bars:
                return self._empty_result(
                    reason="Too many NaN rows after cleaning"
                )

        close  = window["close"].to_numpy(dtype=float)
        high   = window["high"].to_numpy(dtype=float)
        low    = window["low"].to_numpy(dtype=float)
        volume = window["volume"].to_numpy(dtype=float)

        scores = {
            "trend_score":        self.trend_score(close),
            "structure_score":    self.structure_score(high, low),
            "bounce_score":       self.bounce_score(close),
            "breakdown_score":    self.breakdown_score(low, close),
            "distribution_score": self.distribution_score(close, volume),
            "volatility_penalty": self.volatility_penalty(
                close, full_history=full_history
            ),
            "squeeze_risk_penalty": self.squeeze_risk_penalty(
                close, high, low, volume
            ),
        }

        final = (
            c.w_trend         * scores["trend_score"]
            + c.w_structure    * scores["structure_score"]
            + c.w_bounce       * scores["bounce_score"]
            + c.w_breakdown    * scores["breakdown_score"]
            + c.w_distribution * scores["distribution_score"]
            - c.w_vol_penalty     * scores["volatility_penalty"]
            - c.w_squeeze_penalty * scores["squeeze_risk_penalty"]
        )

        scores["final_score"] = round(float(np.clip(final, 0, 100)), 2)
        scores["valid_bars"]  = len(window)
        scores["status"]      = "OK"

        return scores

    # =========================================================
    # PUBLIC ENTRY POINT — FETCH FROM YAHOO FINANCE + SCORE
    # =========================================================
    def fetch_and_calculate(
        self,
        ticker: str,
        long_history_period: str = "1y"
    ) -> dict:

        if not _YFINANCE_AVAILABLE:
            result = self._empty_result(
                reason="yfinance not installed — run: "
                       "pip install yfinance --break-system-packages"
            )
            result["ticker"] = ticker
            result["fetch_status"] = "FAILED — yfinance not available"
            return result

        c = self.config

        try:
            raw = yf.download(
                ticker,
                period=long_history_period,
                progress=False,
                auto_adjust=True
            )
        except Exception as e:
            result = self._empty_result(reason=f"Download error: {e}")
            result["ticker"] = ticker
            result["fetch_status"] = f"FAILED — {str(e)[:80]}"
            return result

        if raw is None or raw.empty:
            result = self._empty_result(reason="No data returned for ticker")
            result["ticker"] = ticker
            result["fetch_status"] = "FAILED — empty response, check ticker symbol"
            return result

        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)

        raw = raw.rename(columns=str.lower)

        required_cols = {"close", "high", "low", "volume"}
        missing = required_cols - set(raw.columns)
        if missing:
            result = self._empty_result(
                reason=f"Downloaded data missing columns: {missing}"
            )
            result["ticker"] = ticker
            result["fetch_status"] = "FAILED — incomplete data from source"
            return result

        full_history = raw[["close", "high", "low", "volume"]].dropna()

        if len(full_history) < c.min_valid_bars:
            result = self._empty_result(
                reason=f"Only {len(full_history)} bars available, "
                       f"need at least {c.min_valid_bars}"
            )
            result["ticker"] = ticker
            result["fetch_status"] = "FAILED — insufficient history"
            return result

        window = full_history.tail(c.lookback)
        result = self.calculate(window, full_history=full_history)
        result["ticker"] = ticker
        result["fetch_status"] = (
            f"OK — {len(full_history)} bars fetched, "
            f"{result['valid_bars']} used in 1M window"
        )

        return result

    # =========================================================
    # PUBLIC ENTRY POINT — BATCH SCAN A WATCHLIST
    # =========================================================
    def scan(
        self,
        tickers: list,
        long_history_period: str = "1y"
    ) -> pd.DataFrame:

        rows = []

        print(f"\nScanning {len(tickers)} tickers for SHORT 1-Month Quality Score...\n")

        for i, ticker in enumerate(tickers, 1):
            print(f"[{i}/{len(tickers)}] {ticker}...", end=" ")
            result = self.fetch_and_calculate(ticker, long_history_period)
            result_row = {"Ticker": ticker, **result}
            rows.append(result_row)
            print(result.get("fetch_status", "—"))

        df = pd.DataFrame(rows)

        if "final_score" in df.columns:
            df = df.sort_values("final_score", ascending=False).reset_index(drop=True)

        return df

    def _empty_result(self, reason: str) -> dict:
        return {
            "trend_score": np.nan,
            "structure_score": np.nan,
            "bounce_score": np.nan,
            "breakdown_score": np.nan,
            "distribution_score": np.nan,
            "volatility_penalty": np.nan,
            "squeeze_risk_penalty": np.nan,
            "final_score": np.nan,
            "valid_bars": 0,
            "status": f"SKIPPED — {reason}",
        }

    # =========================================================
    # 1. TREND — inverted: rewards consistent DOWNtrend
    # =========================================================
    def trend_score(self, close: np.ndarray) -> float:
        """
        Same OLS-on-log-close, t-stat-weighted methodology as the
        long version's trend_score, but scored so that a strong,
        statistically significant NEGATIVE slope produces a high
        score — i.e. measuring decline quality, not advance quality.
        """
        n = len(close)
        x = np.arange(n, dtype=float)
        log_close = np.log(close)

        x_mean = x.mean()
        ss_x = np.sum((x - x_mean) ** 2)
        slope, intercept = np.polyfit(x, log_close, 1)

        fitted = slope * x + intercept
        resid = log_close - fitted
        dof = max(n - 2, 1)
        resid_var = np.sum(resid ** 2) / dof
        se_slope = np.sqrt(resid_var / ss_x) if ss_x > 0 else np.inf

        t_stat = slope / se_slope if se_slope > 0 else 0.0

        ss_tot = np.sum((log_close - log_close.mean()) ** 2)
        ss_res = np.sum(resid ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

        # flip sign of t_stat relative to the long version —
        # a strongly NEGATIVE slope now drives the score up
        t_component = np.tanh(-t_stat / 4.0)
        score = (t_component * r_squared) * 50 + 50

        return float(np.clip(score, 0, 100))

    # =========================================================
    # 2. STRUCTURE — inverted: rewards confirmed LH/LL
    # =========================================================
    def structure_score(self, high: np.ndarray, low: np.ndarray) -> float:
        """
        Same pivot-confirmation methodology as the long version,
        but scores DECREASING swing highs and DECREASING swing
        lows (lower highs, lower lows) as high quality structure.
        """
        order = self.config.pivot_order
        n = len(high)

        if n < (2 * order + 3):
            return 50.0

        swing_high_idx = argrelextrema(high, np.greater_equal, order=order)[0]
        swing_low_idx  = argrelextrema(low,  np.less_equal,    order=order)[0]

        swing_highs = high[swing_high_idx]
        swing_lows  = low[swing_low_idx]

        lh_ratio = np.nan
        ll_ratio = np.nan

        if len(swing_highs) >= 2:
            lh_count = np.sum(np.diff(swing_highs) < 0)
            lh_ratio = lh_count / (len(swing_highs) - 1)

        if len(swing_lows) >= 2:
            ll_count = np.sum(np.diff(swing_lows) < 0)
            ll_ratio = ll_count / (len(swing_lows) - 1)

        ratios = [r for r in (lh_ratio, ll_ratio) if not np.isnan(r)]

        if not ratios:
            return 50.0

        score = np.mean(ratios) * 100
        return float(np.clip(score, 0, 100))

    # =========================================================
    # 3. BOUNCE — NOT pullback_score inverted; new logic
    # =========================================================
    def bounce_score(self, close: np.ndarray) -> float:
        """
        For longs, pullback_score rewards a SHALLOW dip within an
        uptrend as the low-risk entry. The short-side analogue is
        a SHALLOW, ORDERLY BOUNCE within a downtrend — a stock that
        bounces a little, fails, and resumes lower is offering a
        cleaner short entry than one that has shown ZERO bounce at
        all (which is actually a squeeze-risk warning sign, since
        a stock that has fallen in a straight line with no relief
        bounce is statistically more likely to produce a violent
        snap-back than one that has already released some pressure
        via orderly bounces).

        This deliberately does NOT just invert pullback_score's
        "less drawdown = better" logic, because for shorts the
        equivalent of "drawdown" is upward bounce distance from
        the trough, and zero bounce is not the safest case here.
        """
        running_trough = np.minimum.accumulate(close)
        bounce = (close - running_trough) / running_trough   # >= 0

        max_bounce = np.max(bounce)
        mean_bounce = np.mean(bounce)

        if max_bounce < 0.005:
            # essentially no bounce at all anywhere in the window —
            # straight-line decline with no relief — elevated
            # squeeze risk on the NEXT bounce, scored lower than
            # a stock that has already had orderly relief bounces
            depth_score = 55.0
        elif max_bounce <= 0.08:
            # ideal — orderly, contained bounces, trend remains clean
            depth_score = 100.0 - (max_bounce / 0.08) * 15.0   # 85-100
        elif max_bounce <= 0.15:
            # bounce getting large enough to question trend control
            depth_score = 85.0 - ((max_bounce - 0.08) / 0.07) * 35.0  # 50-85
        else:
            # large bounce — trend control deteriorating, squeeze
            # may already be underway
            depth_score = max(0.0, 50.0 - (max_bounce - 0.15) * 200.0)

        # reward LOW average distance from the trough (orderly decline)
        consistency_score = 100.0 * np.exp(-abs(mean_bounce) * 12.0)

        score = 0.65 * depth_score + 0.35 * consistency_score
        return float(np.clip(score, 0, 100))

    # =========================================================
    # 4. BREAKDOWN — inverted breakout_score, same look-ahead fix
    # =========================================================
    def breakdown_score(self, low: np.ndarray, close: np.ndarray) -> float:
        """
        Mirrors the long version's look-ahead bug fix: the prior
        rolling LOW excludes the current bar, so this measures
        genuine breaks BELOW established PRIOR support rather than
        a tautological self-comparison against a minimum that
        already includes today's own low.
        """
        n = len(close)
        if n < 3:
            return 50.0

        prior_low = np.full(n, np.nan)
        running_min = np.inf
        for i in range(n):
            prior_low[i] = running_min
            running_min = min(running_min, low[i])

        valid = ~np.isnan(prior_low) & (prior_low < np.inf)
        if valid.sum() == 0:
            return 50.0

        breakdown = close[valid] < prior_low[valid]
        freq = np.sum(breakdown) / valid.sum()

        if freq <= 0.25:
            score = freq / 0.25 * 100.0
        else:
            score = max(0.0, 100.0 - (freq - 0.25) * 150.0)

        return float(np.clip(score, 0, 100))

    # =========================================================
    # 5. DISTRIBUTION — down-days backed by above-avg volume
    # =========================================================
    def distribution_score(self, close: np.ndarray, volume: np.ndarray) -> float:
        """
        Mirrors volume_score's statistical approach but for the
        sell side: measures whether DOWN-days carry disproportionate
        volume relative to up-days, plus counts genuine distribution
        days (down close on volume above the multiplier threshold).

        Weighted higher in the overall short score (0.24 vs longs'
        0.18) because confirmed institutional distribution matters
        more to short conviction — a price decline without volume
        confirmation is exactly the retail-driven, squeeze-prone
        scenario this framework has repeatedly flagged as the
        weakest basis for a short entry.
        """
        n = len(close)
        if n < 3:
            return 50.0

        avg_vol = np.mean(volume)
        if avg_vol <= 0:
            return 50.0

        daily_ret = np.diff(close) / close[:-1]
        vol_aligned = volume[1:]

        up_mask   = daily_ret > 0
        down_mask = daily_ret < 0

        up_vol_avg   = np.mean(vol_aligned[up_mask])   if up_mask.any()   else 0.0
        down_vol_avg = np.mean(vol_aligned[down_mask]) if down_mask.any() else 0.0

        if up_vol_avg + down_vol_avg == 0:
            return 50.0

        down_vol_share = down_vol_avg / (up_vol_avg + down_vol_avg)

        distribution_days = np.sum(
            (daily_ret < 0) & (vol_aligned > avg_vol * self.config.distribution_vol_mult)
        )
        distribution_ratio = distribution_days / max(n - 1, 1)

        score = (down_vol_share * 60.0) + (distribution_ratio * 100.0 * 0.40)
        return float(np.clip(score, 0, 100))

    # =========================================================
    # 6. VOLATILITY PENALTY — same mechanics, harsher weight
    # =========================================================
    def volatility_penalty(
        self,
        close: np.ndarray,
        full_history: Optional[pd.DataFrame] = None
    ) -> float:
        """
        Identical calculation methodology to the long version
        (annualized vol percentile vs own trailing history when
        available, calibrated absolute bands as fallback). The
        harsher treatment for shorts comes entirely from the
        CONFIG WEIGHT (w_vol_penalty=0.20 vs longs' 0.15), applied
        in calculate() — not from a different formula here. This
        keeps the underlying statistic comparable across both
        scores while letting position-level risk asymmetry between
        longs and shorts live in the weighting, where it belongs.
        """
        returns = np.diff(close) / close[:-1]
        if len(returns) < 2:
            return 0.0

        daily_vol = np.std(returns, ddof=1)
        annualized_vol = daily_vol * np.sqrt(252)

        if full_history is not None and len(full_history) >= self.config.vol_lookback_long:
            hist = full_history.rename(columns=str.lower).tail(
                self.config.vol_lookback_long
            )
            hist_close = hist["close"].to_numpy(dtype=float)
            hist_returns = np.diff(hist_close) / hist_close[:-1]

            window = self.config.lookback
            rolling_vol = pd.Series(hist_returns).rolling(window).std(ddof=1) * np.sqrt(252)
            rolling_vol = rolling_vol.dropna()

            if len(rolling_vol) >= 20:
                percentile = (rolling_vol < annualized_vol).mean()
                penalty = float(np.clip(percentile, 0, 1))
                return penalty

        if annualized_vol <= 0.20:
            penalty = 0.0
        elif annualized_vol <= 0.40:
            penalty = (annualized_vol - 0.20) / 0.20 * 0.5
        elif annualized_vol <= 0.70:
            penalty = 0.5 + (annualized_vol - 0.40) / 0.30 * 0.5
        else:
            penalty = 1.0

        return float(np.clip(penalty, 0, 1))

    # =========================================================
    # 7. SQUEEZE RISK PENALTY — new, no long-side equivalent
    # =========================================================
    def squeeze_risk_penalty(
        self,
        close: np.ndarray,
        high: np.ndarray,
        low: np.ndarray,
        volume: np.ndarray
    ) -> float:
        """
        Genuinely new component with no equivalent in the long-side
        score. Penalizes two independent squeeze-risk markers
        established earlier in this framework's mathematical edge
        analysis of small-cap and thin-liquidity shorts:

        1. Thin RECENT volume relative to the window average —
           low liquidity makes exit difficult if the thesis fails,
           and increases squeeze severity when it does.

        2. Frequency of sharp single-bar reversals (large range
           bars where price closes far from where it opened,
           against the prevailing direction) — repeated violent
           single-bar moves within the window suggest a stock
           prone to gap/squeeze behavior independent of trend
           quality.

        Returns a value in [0, 1] representing penalty SEVERITY.
        """
        n = len(close)
        if n < 5:
            return 0.0

        # --- Marker 1: recent liquidity thinning ---
        recent_window = max(5, n // 4)
        recent_vol = np.mean(volume[-recent_window:])
        full_vol = np.mean(volume)

        if full_vol <= 0:
            liquidity_penalty = 0.0
        else:
            vol_ratio = recent_vol / full_vol
            if vol_ratio >= 0.7:
                liquidity_penalty = 0.0
            else:
                liquidity_penalty = float(np.clip((0.7 - vol_ratio) / 0.7, 0, 1))

        # --- Marker 2: sharp single-bar reversal frequency ---
        bar_range = high - low
        avg_range = np.mean(bar_range)

        if avg_range <= 0:
            reversal_penalty = 0.0
        else:
            close_position = np.where(
                bar_range > 0,
                (close - low) / np.where(bar_range > 0, bar_range, 1),
                0.5
            )
            large_range_mask = bar_range > (avg_range * 1.8)
            squeeze_bars = large_range_mask & (close_position > 0.6)
            squeeze_freq = np.sum(squeeze_bars) / n
            reversal_penalty = float(np.clip(squeeze_freq * 4.0, 0, 1))

        penalty = 0.5 * liquidity_penalty + 0.5 * reversal_penalty
        return float(np.clip(penalty, 0, 1))


# =============================================================
# EXAMPLE USAGE
# =============================================================

if __name__ == "__main__":

    engine = OneMonthShortQualityScore(ShortQualityScoreConfig(lookback=20))




    # ── Test 3 — batch scan a short watchlist ──────────────────
    print("\n" + "=" * 50)
    print(" BATCH SHORT WATCHLIST SCAN")
    print("=" * 50)

    watchlist = df2['Asset'].tolist()
    scan_df = engine.scan(watchlist, long_history_period="1y")

scan_df.rename(columns={"final_score": "qs_1M"}, inplace=True)
scan_df

In [ ]:
## Join with df2
scan_df_cols = ['Ticker','qs_1M']

df3 = df2.merge(
    scan_df[scan_df_cols],
    left_on="Asset",
    right_on="Ticker",
    how="left"
)

# Drop duplicate Ticker column
if "Ticker" in df3.columns:
    df3 = df3.drop(columns=["Ticker"])

print("\n✅ News Sentiment merged into df2 successfully!")
print(f"Final shape: {df3.shape}")

df3.head()

# US Stock Entries (A-Line)

In [ ]:
# SP 500 stocks
# Fetch the Entry_Signal from buy_list
try:
  df4 = df3.copy()
  #df3 = df2[df2['Asset'].isin(top_assets['Ticker'])]
  sp500_stocks_dt = df4[(df4['Type'] == 'Stock') & (df4['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  #df4[(df4['Type'] == 'Stock') & (df4['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = sp500_stocks_dt['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_extended_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_extended_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_sp500_list = sp500_stocks_dt[sp500_stocks_dt["Asset"].isin(final_extended_selection)]

except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_sp500_list = pd.DataFrame({"Asset": ["No Asset available"]})



filtered_sp500_list

## Dutch Lag Cap Stock Entries (Aline Short Entry)

In [ ]:
# AEX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  aex_stocks = df4[(df4['Type'] == 'AS') & (df4['Entry Signal'].isin(['Aline Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = aex_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers

  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_aex_list = aex_stocks[aex_stocks["Asset"].isin(final_aline_selection)]

except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_aex_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_aex_list

## ASX Stock Entries (Aline Short Entry)

In [ ]:
# ASX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  asx_stocks = df4[(df4['Type'] == 'ASX') & (df4['Entry Signal'].isin(['Aline Short Entry','True Trend Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = asx_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers
  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_ASXAL_list = asx_stocks[asx_stocks["Asset"].isin(final_aline_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_ASXAL_list = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_ASXAL_list

## TSX Stock Entries (Aline Short Entry)

In [ ]:
# ASX Small Capstocks
# Fetch the Entry_Signal from buy_list
try:
  tsx_stocks = df4[(df4['Type'] == 'TSX') & (df4['Entry Signal'].isin(['Aline Short Entry']))].reset_index(drop=True)
  # Example usage
  tickers = tsx_stocks['Asset'].tolist()  # Replace with your list of tickers
  ranked_picks = tickers
  final_aline_selection, corr_matrix = get_uncorrelated_picks(tickers, ranked_picks, threshold=0.66)

  #print("Final uncorrelated picks:", final_aline_selection)
  print("\nCorrelation matrix:\n", corr_matrix)
  # Keep only rows where Asset is in filtered
  filtered_TSXAL_list = tsx_stocks[tsx_stocks["Asset"].isin(final_aline_selection)]
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  filtered_TSXAL_list  = pd.DataFrame({"Asset": ["No Asset available"]})

filtered_TSXAL_list